# E3 + free-stream prior - Pressure-only ModalPINN (32 taps)

Same as E3 (32-tap pressure-only) but with `--FreestreamBC` enabled: blends the network's mean velocity mode toward the known free-stream value (u=1, v=0) near the inlet, upstream of the cylinder - not applied downstream/in the wake. Everything else (hyperparameters, tap count, seed) is identical to E3, so the regional evaluation numbers are directly comparable.

**Before running:** Runtime > Change runtime type > T4 GPU.

Use **Runtime > Run all** (not clicking cells one at a time) so the Drive-copy and evaluation steps fire automatically once training finishes, even if you're away when it completes. Cell 2 will prompt you to authorize Google Drive access - approve it.


## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Do this first, before anything else, so results have somewhere durable to land regardless of what happens to this VM later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ModalPINN_results', exist_ok=True)
print('Drive mounted and target folder ready')


## 3. Install Miniconda + Python 3.7 + CUDA 10.0 + cuDNN 7.6.5

In [ ]:
!curl -sL -o /tmp/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash /tmp/miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda create -y -n modalpinn -c conda-forge --override-channels python=3.7 cudatoolkit=10.0 cudnn=7.6.5


## 4. Install TensorFlow-GPU 1.14 and other pinned dependencies

In [ ]:
!/content/miniconda/envs/modalpinn/bin/pip install \
    tensorflow-gpu==1.14.0 numpy==1.17.4 scipy==1.3.2 \
    matplotlib==3.1.1 gputil protobuf==3.11.3


## 5. Verify TensorFlow actually sees the GPU
(Should print `Tesla T4` under Device mapping.)

In [ ]:
import subprocess, os
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/content/miniconda/envs/modalpinn/lib:' + env.get('LD_LIBRARY_PATH', '')
r = subprocess.run(
    ['/content/miniconda/envs/modalpinn/bin/python', '-c',
     "import tensorflow as tf; sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(log_device_placement=True))"],
    capture_output=True, text=True, env=env)
print(r.stdout[-500:])
print(r.stderr[-1500:])


## 6. Write the source files

All three of `ModalPINN_VortexShedding.py`, `Load_train_data_desync.py`, and `NN_functions.py` come from `src/pressure_only/` this time - `NN_functions.py` needed the new `freestream_target` parameter threaded through `out_nn_modes_uv`/`NN_time_uv`, so it's no longer shared unchanged from `src/`.

In [ ]:
%%writefile Load_train_data_desync.py
# -*- coding: utf-8 -*-
"""
This file contains functionsspecific to
-Load_train_data_desync.py:
    Python file containing functions that extract and prepare data for training 
    and validation.
@author: Gaétan Raynaud
"""

# =============================================================================
# Library import
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib.pyplot as plt
from text_flow import read_flow
from reactions_process import extract_reactions

# =============================================================================
# matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


def gen_int_random_points(Npoints,geom,method = 'uniform', disp_plot = False, Delta_r_c = 0.5):
    '''
    Generate Npoints randomly sampled points inside fluid domains defined by 
    Its rectangular shape of width Lx = Lxmax-Lxmin and height Ly = Lymax-Lymin
    The cylinder at position (x_c,y_c) and of radius r_c
    ----
    Return x,y :two arrays of size Npoints 
    containing x and y coordinates of mentionned points
    ----
    Sampling method availables :
        uniform : x ~ U(Lxmin,Lxmax) & y ~ U(Lymin,Lymax)
        y_normal : x ~ U(Lxmin,Lxmax) & y ~ N(0,0.5) \cup [Lymin,Lymax]
        2zones : distribute 80% of data points uniformly in the domain, and 20 in a small region around the cylinder of width Delta_r_c
    ----
    disp_plot : boolean
    If True, display a plot showing the sampling of generated points
    '''
    print('Generating %d points for equations penalization'%(Npoints))
    print('Method = '+method)
    
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    
    x = np.zeros(Npoints)
    y = np.zeros(Npoints)
    
    for j in range(Npoints):
        indomain = False
        
        while indomain == False:
            
            x_test = (Lxmax-Lxmin)*np.random.rand(1)[0] + Lxmin
            
            if method == 'y_normal':
                y_test = np.random.normal(y_c,0.5*(Lymax-Lymin),1)[0]
                
            elif method == '2zones' and np.random.uniform()>0.8:
                r_random = r_c + np.random.uniform()*Delta_r_c
                theta_random = np.random.uniform()*2*np.pi
                x_test = x_c + r_random*np.cos(theta_random)
                y_test = y_c + r_random*np.sin(theta_random)
                
            else:
                y_test = (Lymax-Lymin)*np.random.rand(1)[0] + Lymin
            
            if ((x_test-x_c)**2 + (y_test-y_c)**2 > r_c**2) and y_test < Lymax and y_test > Lymin:
                x[j] = x_test
                y[j] = y_test
                indomain = True

    if disp_plot:
        plt.figure()
        plt.scatter(x,y,c='black',marker='.',s=1.)
        plt.xlabel('$x$')
        plt.ylabel('$y$')
        plt.title('Random training points generation')
        plt.tight_layout()
    
    return x,y





def read_cut_simulation_data(filename_data,geom):
    '''
    Read simulation data results from filename_data file
    Crop data points in the fluid domain defined by Lxmin,Lxmax,Lymin,Lymax
    ----
    Return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    # Step 1 : data loading
    print('Simulation data reading...')
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(filename_data) 
    
    # Step 2 : Selection of points
    
    condition_cut_parts = np.array([np.where(nodes_X[0,:] < Lxmax,True,False), \
                  np.where(nodes_X[0,:] > Lxmin,True,False), \
                  np.where(nodes_Y[0,:] > Lymin,True,False), \
                  np.where(nodes_Y[0,:] < Lymax,True,False)])
    condition_cut = np.all(condition_cut_parts,axis=0)
    
    index_cut = np.argwhere(condition_cut)[:,0]
    
    # Step 3 : cropping
    
    nodes_X = nodes_X[:,index_cut]
    nodes_Y = nodes_Y[:,index_cut]
    Us = Us[:,index_cut]
    Vs = Vs[:,index_cut]
    Ps = Ps[:,index_cut]
    
    print('Reading and cropping simulation data ... ok')
    
    return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps
    

def cut_data_index(index_space,nodes_X, nodes_Y, Us, Vs, Ps):
    '''
    return data nodes_X, nodes_Y, Us, Vs, Ps [:,index_space]
    '''
    nodes_X = nodes_X[:,index_space]
    nodes_Y = nodes_Y[:,index_space]
    Us = Us[:,index_space]
    Vs = Vs[:,index_space]
    Ps = Ps[:,index_space]
    
    return nodes_X, nodes_Y, Us, Vs, Ps
    
    
def cut_simu_cylinder_only(geom,nodes_X, nodes_Y, Us, Vs, Ps, n_taps=30):
    '''
    nodes_X, nodes_Y, Us, Vs, Ps : [Ntime,Nelt] array of scalars
    feom : array containing geom info [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c]
    n_taps : number of pressure taps to select uniformly around the cylinder border
    return data from the points located on the cylinder border
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom

    eps = 1e-5
    r = np.sqrt(np.square(nodes_X[0,:]-x_c) + np.square(nodes_Y[0,:]-y_c))
    delta_r_rc = np.square(r-r_c)

    condition_cut_cyl = np.where(delta_r_rc < eps, True, False)
    index_cylinder = np.argwhere(condition_cut_cyl)[:,0]

    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_cylinder,nodes_X, nodes_Y, Us, Vs, Ps)

    # Select 30 randomly points
    # np.random.shuffle(index_cylinder)
    # index_cylinder = index_cylinder[:30]

    # Select n_taps points that are the nearest from a uniform disposition over the cylinder
    # endpoint=False so that requesting n_taps genuinely yields n_taps distinct physical
    # locations (endpoint=True would place s=0 and s=1 on the same point on the circle)
    s_lin = np.linspace(0.,1.,n_taps,endpoint=False)
    x_points = x_c + r_c*np.cos(2*np.pi*s_lin)
    y_points = y_c + r_c*np.sin(2*np.pi*s_lin)

    index_reduce = 0*x_points
    index_reduce = np.asarray([int(i) for i in index_reduce])
    for k in range(len(x_points)):
        index_reduce[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))

    print('Cylinder taps requested: %d, distinct mesh nodes found: %d' % (n_taps, len(np.unique(index_reduce))))

    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_reduce,nodes_X, nodes_Y, Us, Vs, Ps)

    return nodes_X, nodes_Y, Us, Vs, Ps
    
def cut_simu_pitot_only(geom,nodes_X, nodes_Y, Us, Vs, Ps):
    '''
    nodes_X, nodes_Y, Us, Vs, Ps : [Ntime,Nelt] array of scalars
    feom : array containing geom info [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c] 
    Return data from the points on pitot points locations
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    d = 2*r_c
    # Step 2 : defining the position of wanted points
    N_per_section = 10
    x_points = np.zeros(4*N_per_section)
    y_points = np.zeros(4*N_per_section)
    
    # line 1
    x_points[:N_per_section] = -3.*d*np.ones(N_per_section)
    y_points[:N_per_section] = np.linspace(Lymin,Lymax,N_per_section)
    
    # line 2 post cylinder
    x_points[N_per_section:2*N_per_section] = d*np.ones(N_per_section)
    y_points[N_per_section:2*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)
    
    # line 3 
    x_points[2*N_per_section:3*N_per_section] = 2*d*np.ones(N_per_section)
    y_points[2*N_per_section:3*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)
    
    # line 4
    x_points[3*N_per_section:4*N_per_section] = 3*d*np.ones(N_per_section)
    y_points[3*N_per_section:4*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)
    
    # Step 3 : finding closest point in data
    index_pitot = 0*x_points
    index_pitot = np.asarray([int(i) for i in index_pitot])
    for k in range(len(x_points)):
        index_pitot[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))
    
    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_pitot,nodes_X, nodes_Y, Us, Vs, Ps)
    
    return nodes_X, nodes_Y, Us, Vs, Ps
    

def read_cut_simulation_data_exp_point_and_cylinder(filename_data,geom,n_taps=30):
    '''
    Read simulation data results from filename_data_result
    Pick out data that are the nearest from simulated experimental measurement points
    n_taps : number of pressure taps to select uniformly around the cylinder border
    ----
    Return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    # Step 1 : data loading and cut into studied domain [Lxmin,Lxmax]x[Lymin,Lymax]
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)

    data_pitot = cut_simu_pitot_only(geom,nodes_X, nodes_Y, Us, Vs, Ps)
    #data_pitot = [x_pitot, y_pitot, u_pitot_, v_pitot, p_pitot]

    data_cyl = cut_simu_cylinder_only(geom,nodes_X, nodes_Y, Us, Vs, Ps, n_taps=n_taps)
    #data_cyl = [x_cyl, y_cyl, u_cyl, v_cyl, p_cy]

    return times,data_cyl,data_pitot
    
def read_cut_simulation_data_inlet_points(filename_data,geom):
    '''
    Read simulation data results from filename_data
    Select points from a uniform sampling of location at x = Lxmin and between y = Lymin to y= Lymax on 10 points
    Return times, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet 
    where times [Nt,] list of instants
    and x,y,u,v,p are of size [Nt,10]
    '''    
    
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)
    
    # Select 10 points that are the nearest from a uniform disposition over the inlet
    s_lin = np.linspace(0.,1.,10)
    x_points = Lxmin + 0.*s_lin
    y_points = Lymin + s_lin*(Lymax-Lymin)
    
    index_reduce = 0*x_points
    index_reduce = np.asarray([int(i) for i in index_reduce])
    for k in range(len(x_points)):
        index_reduce[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))
   
    x_inlet, y_inlet, u_inlet, v_inlet, p_inlet = cut_data_index(index_reduce,nodes_X, nodes_Y, Us, Vs, Ps)
    
    
    return times, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet


def data_flatten_cut(x,y,t,u,v,p,Nmes=0):
    '''
    x,y,t,u,v,p : [Nt,Nelts] array
    Nmes (int) : random truncature of data to Nmes. If Nmes= 0, no truncature is performed
    ----
    return x_ft,y_ft,t_ft,u_ft,v_ft,pft flattened and truncated : 1D array of size [Nmes,] or [Nt*Nelts] if Nmes = 0
    '''
    
    index = np.array(range(len(x[0,:])*len(x[:,0])))
    
    if Nmes != 0:
        np.random.shuffle(index)
        index = index[:Nmes]
    
    x = np.ndarray.flatten(x)[index]
    y = np.ndarray.flatten(y)[index]
    t = np.ndarray.flatten(t)[index]
    u = np.ndarray.flatten(u)[index]
    v = np.ndarray.flatten(v)[index]
    p = np.ndarray.flatten(p)[index]
    
    return x,y,t,u,v,p
    
def get_reactions(filename,timemin=-1.,timemax=1e10):
    '''
    Extract forces on cylinder from a .reactions file using extract_reactions()
    Cut time axis between timemin and timemax
    Return times, Fx, Fy
    '''
    times, Fx, Fy, Mz, flag = extract_reactions(filename)
    
    condition_cut_parts = np.array([np.where(times > timemin, True, False), np.where(times < timemax, True, False)])    
    condition_cut = np.all(condition_cut_parts,axis=0)
    
    index_cut = np.argwhere(condition_cut)[:,0]
    
    times = times[index_cut]
    Fx = Fx[index_cut]
    Fy = Fy[index_cut]
    
    return times, Fx, Fy
    
    
def addNoise(x,stdNoise):
    """
    x : 1D np array
    stdNoise float > 0. : standard deviation of Gaussian Noise
    Return x + epsilon, epsilon ~ N(0,std)
    """

    return x + np.random.normal(loc=0.0,scale=stdNoise,size=len(x))
    
    
def training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmin=0.,Tintmax=1e2,cut=True,data_selection='all',desync=False,multigrid=False,Ngrid=10,stdNoise=0.,method_int = '2zones',n_taps=30):
    '''
    cut = True : if True, cut data set to only Nmes points
    if False, keep all the values
    ---
    data_selection (str) :
        if 'all' : returns data randomly sampled in fluid domain in quantity Nmes
        if 'inlet' : returns data at 10 points uniformly separated points at inlet
        if 'cylinder_only' : returns data only from points on the border of the cylinder
        if 'cylinder_pitot' : returns data from cylinder border and on pitot points
        if 'pitot_only'  returns data only at pitot points
    desync = False : (bool) if True, add a uniformly distributed phase shift for measuremnts in pitot and cylinder at each position
    multigrid = False (bool) if True, return a list of size Ngrid, each element containing a sampling of space-time coordinates of size Nint
    Ngrid (int) length of the list of sampled space-time coordinates returned when multigrid = True
    '''
    # Part 1 : border normalised coordinate
    s_train = np.random.rand(Nbc)
    # Part 2 : int points
    
    if multigrid:
        x_int = []
        y_int = []
        t_int = []
        for k in range(Ngrid):
            x_int_temp,y_int_temp = gen_int_random_points(Nint,geom,method = method_int,disp_plot=False)
            x_int.append(x_int_temp)
            y_int.append(y_int_temp)
            t_int.append(Tintmin + (Tintmax-Tintmin)*np.random.rand(Nint))
    else:
        x_int,y_int = gen_int_random_points(Nint,geom,method = 'uniform',disp_plot=False)
        t_int = Tintmin + (Tintmax-Tintmin)*np.random.rand(Nint)
    
    # Part 3 : simulation data points
    
    
    if data_selection == 'all':
        
        Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)
        times_dedouble = np.asarray([t*np.ones(len(nodes_X[0,:])) for t in times])
        if cut:
            xmes,ymes,tmes,umes,vmes,pmes = data_flatten_cut(nodes_X,nodes_Y,times_dedouble,Us,Vs,Ps,Nmes)
        else:
            xmes,ymes,tmes,umes,vmes,pmes = data_flatten_cut(nodes_X,nodes_Y,times_dedouble,Us,Vs,Ps)
    
        return x_int,y_int,t_int,s_train,xmes,ymes,tmes,umes,vmes,pmes
    
    elif data_selection == 'inlet':
        
        times2, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet = read_cut_simulation_data_inlet_points(filename_data,geom)
        t_inlet = np.asarray([t*np.ones(len(x_inlet[0,:])) for t in times2])
        x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet = data_flatten_cut(x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet)
        
        
        return x_int, y_int, t_int, s_train, x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet
    
    else:
        cut = False
        times,data_cyl,data_pitot = read_cut_simulation_data_exp_point_and_cylinder(filename_data,geom,n_taps=n_taps)
        
        # Cylinder data
        xmes_cyl, ymes_cyl, umes_cyl, vmes_cyl, pmes_cyl = data_cyl
        tmes_cyl = np.asarray([t*np.ones(len(xmes_cyl[0,:])) for t in times])
        xmes_cyl, ymes_cyl, tmes_cyl, umes_cyl, vmes_cyl, pmes_cyl = data_flatten_cut(xmes_cyl, ymes_cyl, tmes_cyl, umes_cyl, vmes_cyl, pmes_cyl)
        pmes_cyl = addNoise(pmes_cyl,stdNoise)
        
        # Pitot data
        xmes_pitot, ymes_pitot, umes_pitot, vmes_pitot, pmes_pitot = data_pitot
        
        Nxpitot = 40
        TDesyncMax = 6.06
        if desync:
            #Delta_phi_np_pitot = np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            Delta_t_np_pitot = np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            tmes_pitot = np.asarray([[times[t] + Delta_t_np_pitot[k] for k in range(len(xmes_pitot[t,:]))] for t in range(len(times))])
        else:
            Delta_t_np_pitot = 0.*np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            tmes_pitot = np.asarray([t*np.ones(len(xmes_pitot[0,:])) for t in times])
            
        xmes_pitot, ymes_pitot, tmes_pitot, umes_pitot, vmes_pitot, pmes_pitot = data_flatten_cut(xmes_pitot, ymes_pitot, tmes_pitot, umes_pitot, vmes_pitot, pmes_pitot)
        umes_pitot = addNoise(umes_pitot,stdNoise)
        vmes_pitot = addNoise(vmes_pitot,stdNoise)
        pmes_pitot = addNoise(pmes_pitot,stdNoise)
        
        if data_selection == 'cylinder_only':
            return x_int,y_int,t_int,s_train,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl
        
        elif data_selection == 'pitot_only':

            return x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,Delta_t_np_pitot
        
        elif data_selection == 'cylinder_pitot' :
        
            return x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl,Delta_t_np_pitot
        
        else :
            
            return x_int,y_int,t_int,s_train

def find_pression_static_amont():
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)
    x_target = -4.
    y_target = -4.
    index = np.argmin(np.square(nodes_X[0,:] - x_target)+np.square(nodes_Y[0,:] - y_target))
    plt.figure()
    plt.plot(times,Ps[:,index])
    plt.xlabel('$t$')
    plt.ylabel('Pressure $p$')


In [ ]:
%%writefile ModalPINN_VortexShedding.py
# -*- coding: utf-8 -*-
"""
ModalPINN Python Code
This is the main Python file for performing flow reconstruction using ModalPINN
as described in the paper

    ModalPINN : an extension of Physics-Informed Neural Networks with enforced 
    truncated Fourier decomposition  for periodic flow reconstruction using a 
    limited number of imperfect sensors. 
    G. Raynaud, S. Houde, F. P. Gosselin (2021)

This file contains the main losses functions of the ModalPINN as well as the 
main steps of the training. Nonetheless, it calls functions from 
-Load_train_data_desync.py:
    Python file containing functions that extract and prepare data for training 
    and validation.
-NN_functions.py:
    Python file containing functions specific to
        o neural networks (construction, initialisation),
        o optimisers (calling from scipy or tf interfaces, initialisation, training steps),
        o plots.

This file is designed to be launched on a computationel cluster (initially for 
Compute Canada - Graham server) using the following batch commands:
    #!/bin/bash
    #SBATCH --gres=gpu:t4:1
    #SBATCH --nodelist=gra1337
    #SBATCH --cpus-per-task=2
    #SBATCH --mem=50G
    #SBATCH --job-name=ModalPINN
    #SBATCH --time=0-10:00
    
    module load python/3.7.4
    source ~/ENV/bin/activate
    python ./ModalPINN_VortexShedding.py --Tmax 9 --Nmes 5000 --Nint 50000 --multigrid --Ngrid 5 --NgridTurn 200 --WidthLayer 25 --Nmodes 3 
    deactivate

For each job launched, a folder is created in ./OutputPythonScript and is 
identified by date-time information. In this folder, the content of consol prints 
is saved in a out.txt file alongside other files (mode shapes, various plots...)
including the model itself in a pickle archive.

Please refer to the help for the arguments sent to the parser and to the readme 
for librairies requirements.


@author: Gaétan Raynaud. 
ORCID : orcid.org/0000-0002-2802-7366
email : gaetan.raynaud (at) polymtl.ca
"""

# =============================================================================
# Librairies Import
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import datetime
import os
import pickle
from shutil import copyfile
import sys
import GPUtil
import time
import argparse 
from tensorflow.python.client import device_lib

# Code parts
import NN_functions as nnf
import Load_train_data_desync as ltd

# Link to simulations data 
# In the paper, we used those from Boudina et al. (2020) that can be downloaded 
# at https://zenodo.org/record/5039610
filename_data = 'Data/fixed_cylinder_atRe100'

t0 = time.time()
# =============================================================================
# matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


# =============================================================================
# Preparing the writing of console prints in out.txt
# =============================================================================

class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush() # If you want the output to be visible immediately
    def flush(self) :
        for f in self.files:
            f.flush()

# =============================================================================
# File copy and folder creation
# Here we create a folder containing all the data of this job
# And we copy current python files to keep track of how the job was launched
# =============================================================================
r = int(np.ceil(1000*np.random.rand(1)[0])) # This random number is used in case 2 jobs are launched at the exact same time so that the newly created folders does not merge the one into the other
d = datetime.datetime.now()
pythonfile = os.path.basename(__file__)
repertoire = 'OutputPythonScript/ModalPINN_'+ d.strftime("%Y_%m_%d-%H_%M_%S") + '__' +str(r)
os.makedirs(repertoire, exist_ok=True)
copyfile(pythonfile,repertoire+'/Copy_python_script.py')
copyfile('NN_functions.py',repertoire+'/NN_functions.py')
copyfile('Load_train_data_desync.py',repertoire+'/Load_train_data_desync.py')

f = open(repertoire+'/out.txt', 'w')
original = sys.stdout
sys.stdout = Tee(sys.stdout, f)
print('File copy and stdout ok')


# Print devices available


list_devices = device_lib.list_local_devices()
print('Devices available')
print(list_devices)

# =============================================================================
# Set arguments passed through bash
# =============================================================================

parser = argparse.ArgumentParser()

parser.add_argument('--Tmax',type=float,default=None,help="Define the max time allowed for optimisation (in hours)")
parser.add_argument('--Nmodes',type=int,default=2,help="Number of modes, including zero frequency")
parser.add_argument('--Nmes',type=int,default=5000,help="Number of measurement points to provide for optimisation")
parser.add_argument('--Nint',type=int,default=50000,help="Number of computing points to provide for equation evaluation during optimisation")
parser.add_argument('--LossModes',action="store_true",default=False,help="Use of modal equations during optimisation")
parser.add_argument('--multigrid',action="store_true",default=False,help="Use of multi grid")
parser.add_argument('--Ngrid',type=int,default=1,help="Number of batch for Adam optimization")
parser.add_argument('--NgridTurn',type=int,default=1000,help="Number of iterations between each batch changement")
parser.add_argument('--Noise',type=float,default=0.,help="Define standard deviation of gaussian noise added to measurements")
parser.add_argument('--WidthLayer',type=int,default=20,help="Number of neurons per layer and per mode")
parser.add_argument('--SparseData',action="store_true",default=False,help="if activated, use simulated  measurements data for training. Else use dense data")
parser.add_argument('--DesyncSparseData',action="store_true",default=False,help="if activated (and --SparseData == True), then simulated measurements are randomly made out of synchronisation")
parser.add_argument('--TwoZonesSampling',action="store_true",default=False,help="if activated, the sampling of equation penalisation points is carried out using 2 zones (with more points near the cylinder). Else use a uniform sampling")
parser.add_argument('--PressureOnly',action="store_true",default=False,help="if activated (requires --SparseData), drop pitot (u,v) velocity measurements and train only on cylinder-surface pressure taps.")
parser.add_argument('--NTaps',type=int,default=30,help="Number of pressure taps sampled uniformly around the cylinder border when --SparseData is used.")
parser.add_argument('--Seed',type=int,default=0,help="Seed for numpy and TensorFlow RNGs, for reproducible comparisons across tap counts.")
parser.add_argument('--FreestreamBC',action="store_true",default=False,help="Blend the network's mean velocity mode toward the known free-stream value (u=u_in, v=0) near the inlet, upstream of the cylinder. A second, independent prior alongside the existing cylinder no-slip encoding - not used downstream/in the wake, where the real flow is not free-stream.")


args = parser.parse_args()

if args.PressureOnly and not args.SparseData:
    raise ValueError('--PressureOnly requires --SparseData to also be set.')

print('Args passed to python script')
print('Tmax '+str(args.Tmax)+' (h)')
print('Nmodes %d' % (args.Nmodes))
print('Nmes %d' % (args.Nmes))
print('Nint %d' % (args.Nint))
print('Use Loss Modes : ' + str(args.LossModes))
print('Multigrid : '+str(args.multigrid))
print('Ngrid : '+str(args.Ngrid))
print('Ngrid Turn : '+str(args.NgridTurn))
print('STD Noise : %.2e' % (args.Noise))
print('Neurons per layer and per mode : %d' % (args.WidthLayer))
print('Sparse Data : ' + str(args.SparseData))
print('Desync Sparse Data : ' + str(args.DesyncSparseData))
print('Pressure Only : ' + str(args.PressureOnly))
print('NTaps : %d' % (args.NTaps))
print('Seed : %d' % (args.Seed))
print('Freestream BC : ' + str(args.FreestreamBC))

if args.TwoZonesSampling:
    IntSampling = '2zones'
else:
    IntSampling = 'uniform'

print('Sampling of V_in : '+IntSampling)

# =============================================================================
# Reproducibility: seed numpy and TF graph-level RNG
# =============================================================================
np.random.seed(args.Seed)
tf.compat.v1.set_random_seed(args.Seed)
print('Random seed set to %d' % (args.Seed))

if args.PressureOnly:
    repertoire_new = repertoire + '_Ponly_Ntap%d' % (args.NTaps)
    if args.FreestreamBC:
        repertoire_new = repertoire_new + '_FSBC'
    os.rename(repertoire, repertoire_new)
    repertoire = repertoire_new
    print('Repertoire renamed to '+repertoire)

# =============================================================================
# Physical and geometrical parameters 
# =============================================================================

Re = 100.
Lxmin = -4. 
Lxmax = 8. 
Lx = Lxmax-Lxmin
Lymin = -4. 
Lymax = 4. 
Ly = Lymax-Lymin
x_c = 0. # x-Position of the centre of the cylindre
y_c = 0. # y-Position of the centre of the cylindre
r_c = 0.5 # radius
d = 2.*r_c
u_in = 1. 
rho_0 = 1.

omega_0 = 1.036 #Dimensionless frequency

geom = [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c]


def xbc5(s):
    '''
    Compute cylinders border x coordinate as a function of curvilinear abscissa s \in [0,1]
    input : s (tf tensor, usually of shape [Nbc,1])
    return a tf tensor of the same shape as s
    '''
    return x_c + r_c*tf.cos(2*np.pi*s)
def ybc5(s):
    '''
    Compute cylinders border y coordinate as a function of curvilinear abscissa s \in [0,1]
    input : s (tf tensor, usually of shape [Nbc,1])
    return a tf tensor of the same shape as s
    '''
    return y_c + r_c*tf.sin(2*np.pi*s)

# =============================================================================
# Choix de discretisation
# =============================================================================

Nmodes = args.Nmodes

Nmes = args.Nmes # Number of measurement points in the domain in case of dense data
Nint = args.Nint # Number of points to penalize NS equations in \Omega_f
Nbc = 1000 # Number of points to sample on cylinders norder


multigrid = args.multigrid # If true, Adam optimiser will change of V_in sampling 
# every NgridTurn iterations between the Ngrid generated
Ngrid = args.Ngrid
NgridTurn = args.NgridTurn 

stdNoise = args.Noise # In case of artificially noised data, it defines the 
# standard deviation inputted in the Gaussian distribution

# List of frequencies associated with each mode shapes
# Note that it could be replaced with an arbitrary list of frequencies
# or even tf.Variables() that could be optimized during training
list_omega = np.asarray([k*omega_0 for k in range(Nmodes)]) 

# Structure of each Neural Network that approximate a mode shape
layers = [2,args.WidthLayer*Nmodes,args.WidthLayer*Nmodes,Nmodes]

# =============================================================================
# Training tracking variables
# =============================================================================

global it
global listeErrTimeSerie
global listeErrValidTimeSerie

it=0
listeErrTimeSerie = []
listeErrValidTimeSerie = []

plot_config = False

if args.Tmax==None:
    Tmax = None  #0.5*3600 #8h
else:
    Tmax = 3600*args.Tmax


# =============================================================================
# Placeholders declaration
# In TF<2, one can define placeholders and build an operation graph based on these.
# Values are provided only at the computation in a dictionary tf_dict when running
# session.run(TF quantity that depends on placeholders,feed_dict=TF dictionary containing placeholders values)
# =============================================================================

Nxpitot = 40 # Number of simulated pitot probe locations in the flow (4 sections of 10 points)
Ncyl = 30 # Number of points around the cylinder to simulated pressure probes
Ntimes = 201 # Number of timesteps in simulations data

# Placeholders for V_in (penalization of equations)
x_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Placeholders for general fitting data (especially dense data)
x_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
u_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
v_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
p_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Placeholder for simulated pitot probe
x_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
y_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
t_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
u_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
v_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
p_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1]) #  Not really used since only u and v are used at these locations for training


# Preparing desynchronisation of pitot probe. Especially Used if args.DesyncSparseData == True
Delta_phi_np_pitot = 0.*np.random.uniform(low=0.0,high=2*np.pi/omega_0, size=Nxpitot)

if args.DesyncSparseData:
    Delta_t_tf_pitot = tf.Variable(Delta_t_np_pitot,dtype=tf.float32,shape=[Nxpitot])
else:
    Delta_phi_tf_pitot = tf.constant(Delta_phi_np_pitot,dtype=tf.float32,shape=[Nxpitot])


t_tf_mes_pitot_unflatten = tf.reshape(t_tf_mes_pitot,[Ntimes,Nxpitot])
t_tf_mes_pitot_resync_unflatten = tf.convert_to_tensor([[ t_tf_mes_pitot_unflatten[t,k] - Delta_phi_tf_pitot[k] for k in range(Nxpitot)] for t in range(Ntimes)])
t_tf_mes_pitot_resync = tf.reshape(t_tf_mes_pitot_resync_unflatten,[Ntimes*Nxpitot,1]) 

# Cylindre data for simulated pressure probe
x_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
p_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])


# Border
s_tf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
one_s_tf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Frequencies
w_tf = tf.constant(list_omega,dtype=tf.float32,shape=[Nmodes])


# =============================================================================
# Model construction
# =============================================================================

# Initialisation of weights w and biases b for each variable u, v and p
w_u,b_u = nnf.initialize_NN(layers)
w_v,b_v = nnf.initialize_NN(layers)
w_p,b_p = nnf.initialize_NN(layers)

## For the restoration of a previous model, comment the 3 previous lines and uncomment the 3 following
## Make sure that parameters in parser are correct (Nmodes,WidthLayer...)
# repertoire= 'OutputPythonScript/Name_of_the_folder'
# filename_restore = repertoire + '/DNN2_40_40_2_tanh.pickle' # Attention to change the name of .pickle depending of the NN layers
# w_u,b_u,w_v,b_v,w_p,b_p = nnf.restore_NN(layers,filename_restore)
 

# Known free-stream velocity, used as a prior near the inlet when
# --FreestreamBC is set (see NN_functions.f_freestream_weight/out_nn_modes_uv).
# None when the flag is off, so behaviour is unchanged by default.
freestream_target_u = u_in if args.FreestreamBC else None
freestream_target_v = 0. if args.FreestreamBC else None

def fluid_u(x,y):
    '''
    Compute mode shapes of u
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_uv(x,y,w_u,b_u,geom,freestream_target=freestream_target_u)

def fluid_u_t(x,y,t):
    '''
    Compute u at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_uv(x,y,t,w_u,b_u,geom,omega_0,freestream_target=freestream_target_u)

def fluid_v(x,y):
    '''
    Compute mode shapes of v
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_uv(x,y,w_v,b_v,geom,freestream_target=freestream_target_v)

def fluid_v_t(x,y,t):
    '''
    Compute v at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_uv(x,y,t,w_v,b_v,geom,omega_0,freestream_target=freestream_target_v)

def fluid_p(x,y):
    '''
    Compute mode shapes of p
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_p(x,y,w_p,b_p)

def fluid_p_t(x,y,t):
    '''
    Compute p at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_p(x,y,t,w_p,b_p,omega_0)

# =============================================================================
# Forces on cylinder
# =============================================================================

def force_cylinder_flatten(t):
    '''
    t : tf.float32 tensor shape [Nt,1]  
    ----
    return
    fx_tf,fy_tf :  tf.float32 tensor of shape [Nt,] containing averaged horizontal force on cylinder at time t
    '''
    Nt = int(t.shape[0])
    Ns = 1000 # Number of points to perform the integration over the border
    s_cyl = tf.constant(np.linspace(0.,1.,Ns), dtype = tf.float32, shape = [Ns,1])*tf.transpose(1+0.*t)
    # s_cyl = tf.random.uniform([Ns,1], minval=0., maxval = 1., dtype = tf.float32)*tf.transpose(1+0.*t)
    # Reshaping Space x Times on a same dimension
    s_cyl_r = tf.reshape(s_cyl,[Nt*Ns,1])
    x_cyl_r = tf.reshape(xbc5(s_cyl_r),[Nt*Ns,1]) 
    y_cyl_r = tf.reshape(ybc5(s_cyl_r),[Nt*Ns,1])
    t_cyl = (1.+0*s_cyl)*tf.transpose(t)
    t_cyl_r = tf.reshape(t_cyl,[Nt*Ns,1])
    
    # Computing fluid values along the border
    u = fluid_u_t(x_cyl_r,y_cyl_r,t_cyl_r)
    v = fluid_v_t(x_cyl_r,y_cyl_r,t_cyl_r)
    p = fluid_p_t(x_cyl_r,y_cyl_r,t_cyl_r)
    
    # Computing differentiated quantities
    u_x = tf.gradients(u, x_cyl_r)[0]
    u_y = tf.gradients(u, y_cyl_r)[0]
    u_xx = tf.gradients(u_x, x_cyl_r)[0]
    u_yy = tf.gradients(u_y, y_cyl_r)[0]
    
    v_x = tf.gradients(v, x_cyl_r)[0]
    v_y = tf.gradients(v, y_cyl_r)[0]
    v_xx = tf.gradients(v_x, x_cyl_r)[0]
    v_yy = tf.gradients(v_y, y_cyl_r)[0]
    
    # Computing normal and tangent vectors
    nx_base = - tf.gradients(y_cyl_r, s_cyl_r)[0]
    ny_base = tf.gradients(x_cyl_r, s_cyl_r)[0]
    normalisation = tf.sqrt(tf.square(nx_base) + tf.square(ny_base))
    nx = nx_base/normalisation
    ny = ny_base/normalisation
    
    # Computing local forces elements
    fx_tf_local = -p*nx + 2.*(1./Re)*u_x*nx + (1./Re)*(u_y+v_x)*ny
    fy_tf_local = -p*ny + 2.*(1./Re)*v_y*ny + (1./Re)*(u_y+v_x)*nx
    
    # Reshape to [Ns,Nt]
    fx_tf_local_r2 = tf.reshape(fx_tf_local,[Ns,Nt])
    fy_tf_local_r2 = tf.reshape(fy_tf_local,[Ns,Nt])
    
    # Integrating along the border for every time step
    fx_tf = -2.*np.pi*r_c*tf.reduce_mean(fx_tf_local_r2,axis=0)
    fy_tf = -2.*np.pi*r_c*tf.reduce_mean(fy_tf_local_r2,axis=0)
    
    return fx_tf,fy_tf


# =============================================================================
# Definition of functions for loss
# =============================================================================

def loss_int_mode(x,y):
    '''
    Parameters
    ----------
    x,y : float 32 tensor [Nint,1]
    
    Returns
    -------
    Return a tf.float32 tensor of shape [Nint,1] computing squared errors on modal equations
    '''
    all_u = fluid_u(x,y)
    all_v = fluid_v(x,y)
    all_p = fluid_p(x,y)
    

    one = tf.transpose(0.*x + 1.)
    
    def customgrad(fgrad,xgrad):
        '''
        Input frgad,xgrad : tf.complex64 tensor of shape [1,Nint,N+1] and [1,Nint] resp.
        Return a tf.complex64 tensor df/dx of shape [1,Nint,N+1]
        (tf.gradients does not seem to work with complex values and with f being of order 3... But it is mainly the same thing here)
        '''
        fgrad_xgrad =  [tf.complex(tf.gradients(tf.real(fgrad[:,:,k]), xgrad, grad_ys = one)[0],tf.gradients(tf.imag(fgrad[:,:,k]), xgrad, grad_ys = one)[0]) for k in range(Nmodes)]
        return tf.transpose(tf.convert_to_tensor(fgrad_xgrad), perm=[2,1,0])
    
    all_u_x = customgrad(all_u,x)
    all_u_y = customgrad(all_u,y)
    
    all_v_x = customgrad(all_v,x)
    all_v_y = customgrad(all_v,y)
    
    all_p_x = customgrad(all_p,x)
    all_p_y = customgrad(all_p,y)
    
    all_u_xx = customgrad(all_u_x,x)
    all_u_yy = customgrad(all_u_y,y)
    
    all_v_xx = customgrad(all_v_x,x)
    all_v_yy = customgrad(all_v_y,y)
    
    
    # x axis momentum equation
    f_u = tf.transpose(tf.convert_to_tensor([tf.complex(0.,k*omega_0)*all_u[:,:,k] for k in range(Nmodes)]), perm=[1,2,0])
    f_u += all_p_x
    f_u += (-1./Re)*(all_u_xx + all_u_yy)
    
    f_u_4a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*all_u_x[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_u += tf.transpose(tf.convert_to_tensor(f_u_4a), perm = [1,2,0])
    
    f_u_4b = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*all_u_y[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_u += tf.transpose(tf.convert_to_tensor(f_u_4b), perm = [1,2,0])
    
    f_u_5a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*tf.conj(all_u_x[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5a[-1] = f_u_5a[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5a), perm=[1,2,0])
    
    f_u_5b = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_u[:,:,l-k])*all_u_x[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5b[-1] = f_u_5b[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5b), perm=[1,2,0])

    f_u_5c = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*tf.conj(all_u_y[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5c[-1] = f_u_5c[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5c), perm=[1,2,0])
    
    f_u_5d = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_v[:,:,l-k])*all_u_y[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5d[-1] = f_u_5d[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5d), perm=[1,2,0])    
    
    
    f_u = tf.reduce_sum(nnf.square_norm(f_u), axis=2)
    
    # y axis Momentum equation
    f_v = tf.transpose(tf.convert_to_tensor([tf.complex(0.,k*omega_0)*all_v[:,:,k] for k in range(Nmodes)]), perm=[1,2,0])
    f_v += all_p_y
    f_v += (-1./Re)*(all_v_xx + all_v_yy)
    
    f_v_4a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*all_v_x[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_v += tf.transpose(tf.convert_to_tensor(f_v_4a), perm = [1,2,0])
    
    f_v_4b = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*all_v_y[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_v += tf.transpose(tf.convert_to_tensor(f_v_4b), perm = [1,2,0])
    
    f_v_5a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*tf.conj(all_v_x[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5a[-1] = f_v_5a[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5a), perm=[1,2,0])
    
    f_v_5b = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_u[:,:,l-k])*all_v_x[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5b[-1] = f_v_5b[-2]*0.  #quand k=N, k+1 > N
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5b), perm=[1,2,0])

    f_v_5c = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*tf.conj(all_v_y[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5c[-1] = f_v_5c[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5c), perm=[1,2,0])
    
    f_v_5d = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_v[:,:,l-k])*all_v_y[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5d[-1] = f_v_5d[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5d), perm=[1,2,0])    
    

    f_v = tf.reduce_sum(nnf.square_norm(f_v), axis=2)
    
    
    # Mass conservation equation
    div_u = all_u_x + all_v_y
    div_u = tf.reduce_sum(nnf.square_norm(div_u), axis=2)
    
    return div_u + f_u + f_v


def loss_int_time(x,y,t):
    '''
    Parameters
    ----------
    x,y,t : tf.float 32 tensor [Nint,1]

    Returns
    -------
    Return [Nint,1] tensor containing squared error on NS equations
    '''
    u = fluid_u_t(x,y,t)
    v = fluid_v_t(x,y,t)
    p = fluid_p_t(x,y,t)
    
    u_t = tf.gradients(u,t)[0]
    v_t = tf.gradients(v,t)[0]
    
    u_x = tf.gradients(u, x)[0]
    u_y = tf.gradients(u, y)[0]
    u_xx = tf.gradients(u_x, x)[0]
    u_yy = tf.gradients(u_y, y)[0]
    
    v_x = tf.gradients(v, x)[0]
    v_y = tf.gradients(v, y)[0]
    v_xx = tf.gradients(v_x, x)[0]
    v_yy = tf.gradients(v_y, y)[0]
    
    p_x = tf.gradients(p, x)[0]
    p_y = tf.gradients(p, y)[0]

    f_u = u_t + (u*u_x + v*u_y) + p_x - (1./Re)*(u_xx + u_yy) 
    f_v = v_t + (u*v_x + v*v_y) + p_y - (1./Re)*(v_xx + v_yy)
    div_u = u_x + v_y
    
    return tf.square(f_u)+tf.square(f_v)+tf.square(div_u)


def loss_mes(xmes,ymes,tmes,umes,vmes,pmes):
    '''
    xmes,ymes,tmes,umes,vmes,pmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements 
    '''
    u_DNN = fluid_u_t(xmes,ymes,tmes)
    v_DNN = fluid_v_t(xmes,ymes,tmes)
    p_DNN = fluid_p_t(xmes,ymes,tmes)
    
    return tf.square(u_DNN-umes) + tf.square(v_DNN-vmes) + tf.square(p_DNN-pmes)

def loss_mes_uv(xmes,ymes,tmes,umes,vmes):
    '''
    xmes,ymes,tmes,umes,vmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements of velocity
    '''
    u_DNN = fluid_u_t(xmes,ymes,tmes)
    v_DNN = fluid_v_t(xmes,ymes,tmes)
    
    return tf.square(u_DNN-umes) + tf.square(v_DNN-vmes)

def loss_mes_p(xmes,ymes,tmes,pmes):
    '''
    xmes,ymes,tmes,pmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements of pressure
    '''
    p_DNN = fluid_p_t(xmes,ymes,tmes)
    
    return tf.square(p_DNN-pmes)


def loss_BC(s):
    '''
    Return error on u=v=0 on cylinder border for each mode
    Input s : [Nbc,1] tf.float32 tensor of coordinates \in [0,1]
    Output : [] tf.float32 real positive number
    '''    
    x = xbc5(s)
    y = ybc5(s)
    u_k = fluid_u(x,y)
    v_k = fluid_v(x,y)
    
    err = tf.convert_to_tensor([nnf.square_norm(u_k[0,:,k]) + nnf.square_norm(v_k[0,:,k]) for k in range(Nmodes)])
    
    return tf.reduce_sum(tf.reduce_mean(err,axis=1))


# =============================================================================
# Training loss creation
# =============================================================================

# Wrap error on modal equations
Loss_int_mode_wrap = tf.reduce_mean(loss_int_mode(x_tf_int, y_tf_int))

# Wrap error on physical equations
Loss_int_time_wrap = tf.reduce_mean(loss_int_time(x_tf_int, y_tf_int ,t_tf_int))

# Wrap error on (u,v,p) measurements
Loss_dense_mes = tf.reduce_mean(loss_mes(x_tf_mes,y_tf_mes,t_tf_mes,u_tf_mes,v_tf_mes,p_tf_mes))

# Wrap error on (u,v) measurements at simulated pitot probes locations
Loss_mes_pitot = tf.reduce_mean(loss_mes_uv(x_tf_mes_pitot,y_tf_mes_pitot,t_tf_mes_pitot_resync,u_tf_mes_pitot,v_tf_mes_pitot))
Loss_mes_pitot_desync = tf.reduce_mean(loss_mes_uv(x_tf_mes_pitot,y_tf_mes_pitot,t_tf_mes_pitot,u_tf_mes_pitot,v_tf_mes_pitot))

# Wrap error on pressure measurement around cylindre border
Loss_mes_cyl = tf.reduce_mean(loss_mes_p(x_tf_mes_cyl,y_tf_mes_cyl,t_tf_mes_cyl,p_tf_mes_cyl))

# Simulated experimental losses
if args.PressureOnly:
    # Pressure-only mode: cylinder-surface pressure taps only, pitot velocity dropped entirely
    Loss_mes_exp = Loss_mes_cyl
else:
    Loss_mes_exp = Loss_mes_pitot + Loss_mes_cyl

if args.SparseData:
    Loss_mes = Loss_mes_exp
else: # Dense measurements are used for training
    Loss_mes = Loss_dense_mes

if args.LossModes:
    Loss = Loss_int_mode_wrap + Loss_mes
else: #Physical equations are used instead of modal equations
    Loss = Loss_int_time_wrap + Loss_mes

# =============================================================================
# Optimizer configuration
# =============================================================================

opt_LBFGS = nnf.declare_LBFGS(Loss)

opt_Adam = nnf.declare_Adam(Loss, lr=1e-5)

sess = nnf.declare_init_session()


# =============================================================================
# GPU use before loading data
# =============================================================================
print('GPU use before loading data')
GPUtil.showUtilization()

# =============================================================================
# Data set preparation
# =============================================================================

if args.SparseData:
    # Let's load data only at locations defined for simulated measurements
    print('Loading Sparse Data')
    
    x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl,Delta_phi_np_pitot_applied = ltd.training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmax=1e2,data_selection = 'cylinder_pitot',desync=args.DesyncSparseData, multigrid=multigrid,Ngrid=Ngrid,stdNoise=stdNoise,method_int = IntSampling, n_taps=args.NTaps)
    Ncyl = len(xmes_cyl)
    Npitot = len(xmes_pitot)
    Tmin = 400.
    
    
    if multigrid:
        tf_dict = []
        for k in range(Ngrid):
            tf_dict_temp = {x_tf_int : np.reshape(x_int[k],(Nint,1)),
             y_tf_int : np.reshape(y_int[k],(Nint,1)),
             t_tf_int : np.reshape(t_int[k],(Nint,1)),
             s_tf : np.reshape(s_train,(Nbc,1)),
             x_tf_mes_cyl : np.reshape(xmes_cyl,(Ncyl,1)),
             y_tf_mes_cyl : np.reshape(ymes_cyl,(Ncyl,1)),
             p_tf_mes_cyl : np.reshape(pmes_cyl,(Ncyl,1)),
             t_tf_mes_cyl : np.reshape(tmes_cyl,(Ncyl,1)),
             x_tf_mes_pitot : np.reshape(xmes_pitot,(Npitot,1)),
             y_tf_mes_pitot : np.reshape(ymes_pitot,(Npitot,1)),
             u_tf_mes_pitot : np.reshape(umes_pitot,(Npitot,1)),
             v_tf_mes_pitot : np.reshape(vmes_pitot,(Npitot,1)),
             p_tf_mes_pitot : np.reshape(pmes_pitot,(Npitot,1)),
             t_tf_mes_pitot : np.reshape(tmes_pitot,(Npitot,1)),
             }
            tf_dict.append(tf_dict_temp)
        
    else:      
        tf_dict = {x_tf_int : np.reshape(x_int,(Nint,1)),
             y_tf_int : np.reshape(y_int,(Nint,1)),
             t_tf_int : np.reshape(t_int,(Nint,1)),
             s_tf : np.reshape(s_train,(Nbc,1)),
             x_tf_mes_cyl : np.reshape(xmes_cyl,(Ncyl,1)),
             y_tf_mes_cyl : np.reshape(ymes_cyl,(Ncyl,1)),
             p_tf_mes_cyl : np.reshape(pmes_cyl,(Ncyl,1)),
             t_tf_mes_cyl : np.reshape(tmes_cyl,(Ncyl,1)),
             x_tf_mes_pitot : np.reshape(xmes_pitot,(Npitot,1)),
             y_tf_mes_pitot : np.reshape(ymes_pitot,(Npitot,1)),
             u_tf_mes_pitot : np.reshape(umes_pitot,(Npitot,1)),
             v_tf_mes_pitot : np.reshape(vmes_pitot,(Npitot,1)),
             p_tf_mes_pitot : np.reshape(pmes_pitot,(Npitot,1)),
             t_tf_mes_pitot : np.reshape(tmes_pitot,(Npitot,1))
             }

else:
    print('Loading Dense Data')
    x_int,y_int,t_int,s_train,xmes,ymes,tmes,umes,vmes,pmes = ltd.training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmax=1e2,data_selection = 'all',desync=False, multigrid=multigrid,Ngrid=Ngrid,stdNoise=stdNoise,cut=True,method_int=IntSampling)
    Nmes = len(xmes)
    Tmin = 400.
    
    if multigrid:
        tf_dict = []
        for k in range(Ngrid):
            tf_dict_temp = {x_tf_int : np.reshape(x_int[k],(Nint,1)),
              y_tf_int : np.reshape(y_int[k],(Nint,1)),
              t_tf_int : np.reshape(t_int[k],(Nint,1)),
              s_tf : np.reshape(s_train,(Nbc,1)),
              x_tf_mes : np.reshape(xmes,(Nmes,1)),
              y_tf_mes : np.reshape(ymes,(Nmes,1)),
              p_tf_mes : np.reshape(pmes,(Nmes,1)),
              t_tf_mes : np.reshape(tmes,(Nmes,1)),
              u_tf_mes : np.reshape(umes,(Nmes,1)),
              v_tf_mes : np.reshape(vmes,(Nmes,1))
              }
            tf_dict.append(tf_dict_temp)
        
    else:      
        tf_dict = {x_tf_int : np.reshape(x_int,(Nint,1)),
              y_tf_int : np.reshape(y_int,(Nint,1)),
              t_tf_int : np.reshape(t_int,(Nint,1)),
              s_tf : np.reshape(s_train,(Nbc,1)),
              x_tf_mes : np.reshape(xmes,(Nmes,1)),
              y_tf_mes : np.reshape(ymes,(Nmes,1)),
              p_tf_mes : np.reshape(pmes,(Nmes,1)),
              t_tf_mes : np.reshape(tmes,(Nmes,1)),
              u_tf_mes : np.reshape(umes,(Nmes,1)),
              v_tf_mes : np.reshape(vmes,(Nmes,1))
              }
    

# Validation data set loading
# We extract 10 times more points for both dense measurements and equation penalisation
print('Loading validation data set')

x_int_valid,y_int_valid,t_int_valid,s_train,xmes_valid,ymes_valid,tmes_valid,umes_valid,vmes_valid,pmes_valid = ltd.training_dict(10*Nmes,10*Nint,Nbc,filename_data,geom,Tintmax=1e2,cut=True,method_int='uniform')
Nmesvalid = len(xmes_valid)

tf_dict_valid = {x_tf_int : np.reshape(x_int_valid,(10*Nint,1)),
     y_tf_int : np.reshape(y_int_valid,(10*Nint,1)),
     t_tf_int : np.reshape(t_int_valid,(10*Nint,1)),
     s_tf : np.reshape(s_train,(Nbc,1)),
     x_tf_mes : np.reshape(xmes_valid,(Nmesvalid,1)),
     y_tf_mes : np.reshape(ymes_valid,(Nmesvalid,1)),
     u_tf_mes : np.reshape(umes_valid,(Nmesvalid,1)),
     v_tf_mes : np.reshape(vmes_valid,(Nmesvalid,1)),
     p_tf_mes : np.reshape(pmes_valid,(Nmesvalid,1)),
     t_tf_mes : np.reshape(tmes_valid,(Nmesvalid,1))}


# =============================================================================
# GPU use after loading data
# =============================================================================
print('GPU use after loading data')
GPUtil.showUtilization()


# =============================================================================
# Training
# =============================================================================

nnf.print_bar()
t1 = time.time()
print('Start training after %d s'%(t1-t0))

print('Start L-BFGS-B training')
List_it_loss_LBFGS,List_it_loss_valid_LBFGS = nnf.model_train_scipy(opt_LBFGS,sess,Loss,tf_dict[0],List_loss = True,tf_dict_valid=tf_dict_valid,loss_valid = Loss_dense_mes)

t2 = time.time()
print('L-BFGS-B training ended after %d s'%(t2-t1))

print('Start Adam training')
# Here Adam training is stopped if it reaches a time limit AdamTmax, or number of iterations Nit or if training loss goes under tolAdam
AdamTmax = Tmax-(t2-t0)
List_it_loss_Adam,List_it_loss_valid_Adam = nnf.model_train_Adam(opt_Adam,sess,Loss,liste_tf_dict=tf_dict,Nit=1e5,tolAdam=1e-5,it=it,itdisp=100,maxTime=AdamTmax,multigrid=multigrid,NgridTurn=NgridTurn,List_loss = True,tf_dict_valid=tf_dict_valid,loss_valid = Loss_dense_mes)
t3 = time.time()
print('Adam training ended after %d s'%(t3-t2))

# =============================================================================
# GPU use after training
# =============================================================================
print('GPU use after training')
GPUtil.showUtilization()
print('End of training')

# =============================================================================
# Print residuals errors and losses
# =============================================================================

nnf.print_bar()
print('Error details')
nnf.print_bar()

if not(multigrid):
    tf_dict = [tf_dict]

print('')
nnf.tf_print('Border',loss_BC(s_tf),sess,tf_dict[0])
nnf.tf_print('Loss eqs. modes',Loss_int_mode_wrap,sess,tf_dict[0])
nnf.tf_print('Loss eqs. int time',Loss_int_time_wrap,sess,tf_dict[0])
nnf.tf_print('Loss mesures training',Loss_mes,sess,tf_dict[0])
nnf.tf_print('Loss mesures validation',Loss_dense_mes,sess,tf_dict_valid)
if args.SparseData:
    nnf.tf_print('Loss mes pitot (component)',Loss_mes_pitot,sess,tf_dict[0])
    nnf.tf_print('Loss mes cyl (component)',Loss_mes_cyl,sess,tf_dict[0])
    
if args.DesyncSparseData:
    
    def r_div_eucli(a,b):
        '''
        a,b real numbers
        return r with a = n*b + r, n (int) and -b/2 <= r < b/2
        '''
        rtemp = a%b
        return np.where(rtemp>0.5*b,rtemp-b,rtemp)

    
    print('Validation Resync')
    Delta_phi_tf_pitot_found_o = sess.run(Delta_phi_tf_pitot)
    err_rms_resync = np.sqrt(np.mean(np.square((r_div_eucli(Delta_phi_tf_pitot_found_o-Delta_phi_np_pitot_applied,2*np.pi/omega_0)))))
    err_rms_resync_normalized = err_rms_resync/np.sqrt(np.mean(np.square(Delta_phi_np_pitot_applied)))
    print('Err RMS Resynchro : %.3e'%(err_rms_resync))
    print('Err RMS Resynchro normalized : %.3e'%(err_rms_resync_normalized))
    
    # Plot répartition des  erreurs de resyncro
    xpitot = np.reshape(xmes_pitot,[Ntimes,Nxpitot])[0,:]
    ypitot = np.reshape(ymes_pitot,[Ntimes,Nxpitot])[0,:]
    err_resync_pitot = r_div_eucli(Delta_phi_tf_pitot_found_o-Delta_phi_np_pitot_applied,2*np.pi/omega_0)
    
    size_resync = np.log10(err_resync_pitot)
    
    plt.figure()
    plt.scatter(xpitot,ypitot,c=np.log10(err_resync_pitot),marker='o',s=1.+size_resync)
    plt.colorbar()
    plt.scatter(xmes_cyl,ymes_cyl,c='black',marker='.',s=1.)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.axis('equal')
    plt.xlim((Lxmin,Lxmax))
    plt.ylim((Lymin,Lymax))
    plt.title('Synchronisation error - log')
    plt.tight_layout()
    plt.savefig(repertoire+'/resync_err.png')
    plt.close()
    


# =============================================================================
# Save NN Model coefficients in a pickle archive
# =============================================================================

print('Saving NN Model...')

str_layers_fluid = [str(j) for j in layers]
filename_fluid = repertoire + '/DNN' + '_'.join(str_layers_fluid) + '_tanh.pickle'

Data_fluid = sess.run([w_u,b_u,w_v,b_v,w_p,b_p])
pcklfile_fluide = open(filename_fluid,'ab+')
pickle.dump(Data_fluid,pcklfile_fluide)
pcklfile_fluide.close()
print('Model exported in '+repertoire)

# =============================================================================
# Save convergence history
# =============================================================================

print('Saving convergence history...')

filename_hist = repertoire + '/Convergence_history.pickle'

Data_loss_history = [List_it_loss_LBFGS,List_it_loss_valid_LBFGS,List_it_loss_Adam,List_it_loss_valid_Adam]
pckl_hist = open(filename_hist,'ab+')
pickle.dump(Data_loss_history,pckl_hist)
pckl_hist.close()
print('History exported in '+repertoire)

plt.figure()
plt.scatter(np.array(List_it_loss_LBFGS)[:,0],np.array(List_it_loss_LBFGS)[:,1],label='LBFGS train',marker='.',s=1.,c='red')
# plt.scatter(np.array(List_it_loss_valid_LBFGS)[:,0],np.array(List_it_loss_valid_LBFGS)[:,1],label='LBFGS valid',marker='.',s=1.,c='pink')
# Validation loss does not seem to be accessible during L-BFGS-B training. It returns constant values
plt.scatter(np.array(List_it_loss_Adam)[:,0]+np.max(np.array(List_it_loss_LBFGS)[:,0]),np.array(List_it_loss_Adam)[:,1],label='Adam train',marker='.',s=1.,c='blue')
plt.scatter(np.array(List_it_loss_valid_Adam)[:,0]+np.max(np.array(List_it_loss_LBFGS)[:,0]),np.array(List_it_loss_valid_Adam)[:,1],label='Adam valid',marker='.',s=1.,c='green')
plt.xlabel('Iterations')
plt.ylabel('Error')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.savefig(repertoire+'/Convergence_history.png')
plt.close()



# =============================================================================
# Plot of modal shapes
# =============================================================================


for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_u(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='u Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/u_mode_'+str(k)+'.png')
    plt.close()
    


for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_v(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='v Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/v_mode_'+str(k)+'.png')
    plt.close()

for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_p(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='p Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/p_mode_'+str(k)+'.png')
    plt.close()


# =============================================================================
# Comparison at a given timestep between modalPINN and simulations data
# =============================================================================
inst = 16

Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = ltd.read_cut_simulation_data(filename_data,geom)

tf_dict_compare = {
    x_tf_mes : np.reshape(nodes_X[0,:],(len(nodes_X[0,:]),1)),
    y_tf_mes : np.reshape(nodes_Y[0,:],(len(nodes_Y[0,:]),1)),
    t_tf_mes : np.reshape(times[inst]*np.ones(len(nodes_X[0,:])),(len(nodes_Y[0,:]),1)),
    u_tf_mes : np.reshape(Us[inst,:],(len(nodes_X[0,:]),1))
    }

suptitle='u difference at t = '+'{0:.2f}'.format(times[inst])

nnf.tf_plot_compare_3plot(x_tf_mes,y_tf_mes,u_tf_mes,fluid_u_t(x_tf_mes,y_tf_mes,t_tf_mes),sess,xlabel='$x$',ylabel='$y$',title1='Exact',title2='ModalPINN',suptitle='',tf_dict=tf_dict_compare)
plt.savefig(repertoire+'/diff_u_t_'+'{0:.2f}'.format(times[inst])+'.png')





In [ ]:
%%writefile NN_functions.py
# -*- coding: utf-8 -*-
"""
This file contains functions specific to
        o neural networks (construction, initialisation),
        o optimisers (calling from scipy or tf interfaces, initialisation, training steps),
        o plots.
@author: Gaétan Raynaud
"""

# =============================================================================
# Libraries
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib.pyplot as plt
import pickle
import time

# =============================================================================
# Matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


# =============================================================================
# Functions for defining, restoring and initialising neural networks
# =============================================================================

# True --> z = w1*exp(i*w2)
# False --> z = w1 + i*w2
complex_value_exp = True  

def initialize_NN(layers,name_nn=''):        
    '''
    Initialize a complex neural network which structure is defined by layers
    Input layers : list of integers defining the width of each layer. 
                   The number of elements in layers defines the depth of the NN
    Return weights : list of matrices filled with tf.complex64 variables
           biases : list of vectors filled with tf.complex64 variables
    '''
    weights = []
    biases = []
    num_layers = len(layers) 
    for l in range(0,num_layers-1):
        W_1 = xavier_init(size=[layers[l], layers[l+1]],name_w = 'weights_'+name_nn+str(l))
        W_2 = xavier_init(size=[layers[l], layers[l+1]],name_w = 'weights_'+name_nn+str(l))

        b_1 = tf.Variable(tf.zeros([1,layers[l+1]], dtype=tf.float32), dtype=tf.float32, name ='biases_'+name_nn+str(l))
        b_2 = tf.Variable(tf.zeros([1,layers[l+1]], dtype=tf.float32), dtype=tf.float32, name ='biases_'+name_nn+str(l))
        
        if complex_value_exp:
            W = tf.complex(W_1, 0.)*tf.exp(tf.complex(0., W_2))
            b = tf.complex(b_1, 0.)*tf.exp(tf.complex(0., b_2))
        else :
            W = tf.complex(W_1, W_2)
            b = tf.complex(b_1, b_2)
        
        weights.append(W)
        biases.append(b)     
        
    return weights, biases

def restore_one_NN(layers,w_value,b_value,tf_as_constant=False):
    '''
    input 
    layers : list of integers describing the structure of the NN
    w_value,b_value : list of values of the tensors coefficients
    tf_as_constant : bool (False) : if True, construct directly tf.comple64 coefficients as tf.constant
    If False, construct tf.Variables as tf.float32 and then join them according to complex_value_exp (bool) policy
    ----
    return
    weights and biases tensors variables initialised with the given values
    '''
    weights = []
    biases = []
    num_layers = len(layers) 
    for l in range(0,num_layers-1):
        if tf_as_constant:
            W = tf.constant(w_value[l],dtype=tf.complex64,shape=[layers[l],layers[l+1]])
            b = tf.constant(b_value[l],dtype=tf.complex64,shape=[1,layers[l+1]])
        
        else:
            if complex_value_exp:
                W_1 = tf.Variable(tf.math.abs(w_value[l]),dtype=tf.float32,shape=[layers[l],layers[l+1]])
                W_2 = tf.Variable(tf.math.angle(w_value[l]),dtype=tf.float32,shape=[layers[l],layers[l+1]])
                W = tf.complex(W_1, 0.)*tf.exp(tf.complex(0., W_2)) 
                
                b_1 = tf.Variable(tf.math.abs(b_value[l]),dtype=tf.float32,shape=[1,layers[l+1]])
                b_2 = tf.Variable(tf.math.angle(b_value[l]),dtype=tf.float32,shape=[1,layers[l+1]])
                b = tf.complex(b_1, 0.)*tf.exp(tf.complex(0., b_2))
                
            else:
                W_1 = tf.Variable(tf.math.real(w_value[l]),dtype=tf.float32,shape=[layers[l],layers[l+1]])
                W_2 = tf.Variable(tf.math.imag(w_value[l]),dtype=tf.float32,shape=[layers[l],layers[l+1]])
                W = tf.complex(W_1,W_2)
    
                b_1 = tf.Variable(tf.math.real(b_value[l]),dtype=tf.float32,shape=[1,layers[l+1]])
                b_2 = tf.Variable(tf.math.imag(b_value[l]),dtype=tf.float32,shape=[1,layers[l+1]])
                b = tf.complex(b_1,b_2)
        weights.append(W)
        biases.append(b)
    return weights,biases
            

def restore_NN(layers,filename_restore,tf_as_constant=False):

    '''
    Restore u, v and p ModalPINN models 
    Input layers : list of each layers width
          filename_restore (str) : location of the pickle archive where the values are stored
          tf_as_constant (bool) : if True, model's parameters are initialised as tf.constant 
                                  (and are therefore fixed). Else, they are set as 
                                  tf.variable and can be trained once again.
    Return the weights and biases 
    '''    

    file = open(filename_restore,'rb')
    w_u_value,b_u_value,w_v_value,b_v_value,w_p_value,b_p_value = pickle.load(file)
    file.close()

    w_u,b_u = restore_one_NN(layers,w_u_value,b_u_value,tf_as_constant)
    w_v,b_v = restore_one_NN(layers,w_v_value,b_v_value,tf_as_constant)
    w_p,b_p = restore_one_NN(layers,w_p_value,b_p_value,tf_as_constant)
    
    return w_u,b_u,w_v,b_v,w_p,b_p
        
        

def xavier_init(size,name_w):
    '''
    Initialisation of weights using xavier init.
    This function comes from Raissi et al. (2019)
    '''
    in_dim = size[0]
    out_dim = size[1]        
    xavier_stddev = np.sqrt(2/(in_dim + out_dim))
    return tf.Variable(tf.random.truncated_normal([in_dim, out_dim], stddev=xavier_stddev, dtype=tf.float32), dtype=tf.float32, name = name_w)


def neural_net(X, weights, biases):
    '''
    Construct one neural network as a succession of affine transformation and 
    non-linear functions (here sigma = tanh)
    Input : tf.tensor X, weights and biases that define the model
    Output: tf.tensor Y
    '''
    H = X
    num_layers = len(weights) + 1
    for l in range(0,num_layers-2):
        W = weights[l]
        b = biases[l]
        H = tf.tanh(tf.add(tf.matmul(H, W), b))
    W = weights[-1]
    b = biases[-1]
    Y = tf.add(tf.matmul(H, W), b)
    return Y


def f_BC5(x, y, geom, fact=5.):
    '''
    return real tensor of same size than x and y
    equal to zero on the cylinder border
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    r = tf.sqrt(tf.square(x-x_c) + tf.square(y-y_c)) - r_c
    return tf.tanh(fact*r)

def f_freestream_weight(x, x_transition=-2., gamma=3.):
    '''
    Blending weight for an inlet free-stream prior: ~1 upstream of the
    cylinder (near the inlet, where the real flow genuinely is undisturbed
    free-stream), ~0 near/after the cylinder and downstream (where the
    real flow is still inside the wake and should NOT be forced toward
    free-stream). Only depends on x, not y or the cylinder radius, since
    the transition is a streamwise one, not a distance-from-cylinder one.
    Input x : [Nint,1] tf.float32 tensor
    Output : [Nint,1] tf.float32 tensor in (0,1)
    '''
    return 0.5 * (1. - tf.tanh(gamma * (x - x_transition)))

def out_nn_modes_uv(x,y,weights,biases,geom,freestream_target=None):
    '''
    Return Nmode complex modes shapes of DNN defined with weights and biases
    Prior dictionary f_BC5 is applied so that each mode shape verifies =0 on cylinder's border
    If freestream_target is not None, the mean mode (k=0) is additionally blended
    toward that known constant near the inlet (see f_freestream_weight) - a second,
    independent prior alongside f_BC5, not a replacement for it.
    Input x,y : [Nint,1] tf.float32 tensor
    Output shape : [1,Nint,Nmode] tf.complex64 tensor
    '''
    xint = tf.complex(x,0.)
    yint = tf.complex(y,0.)
    out_nn = neural_net(tf.transpose(tf.stack([xint,yint])),weights,biases)
    Nmode = int(out_nn[0,0,:].shape[0])
    fbc5c = tf.complex(f_BC5(x,y,geom)[:,0],0.)
    modes = []
    for k in range(Nmode):
        mode_k = fbc5c*out_nn[:,:,k]
        if k == 0 and freestream_target is not None:
            w = tf.complex(f_freestream_weight(x)[:,0], 0.)
            mode_k = w*tf.complex(freestream_target, 0.) + (1.-w)*mode_k
        modes.append(mode_k)
    t_parts = tf.convert_to_tensor(modes)
    return tf.transpose(t_parts,perm=[1,2,0])

def out_nn_modes_p(x,y,weights,biases):
    '''
    Return Nm complex modes of dnn defined with weights and biases
    Input x,y : [Nint,1] real tf.float32 tensor
    Output shape : [1,Nint,Nmode] tf.complex64 tensor
    '''
    xint = tf.complex(x,0.)
    yint = tf.complex(y,0.)
    out_nn = neural_net(tf.transpose(tf.stack([xint,yint])),weights,biases)
    Nmode = int(out_nn[0,0,:].shape[0])
    t_parts = tf.convert_to_tensor([out_nn[:,:,k] for k in range(Nmode)])
    return tf.transpose(t_parts,perm=[1,2,0])


def NN_time_uv(x,y,t,weights,biases,geom,omega_0,trunc_mode=None,freestream_target=None):
    '''
    x,y,t : [Nint,1] tf.float32 tensors, list of coordinates (x,t) where to compute u or v(x,y,t)
    omega_0 : fondamental frequency
    Output [Nint,1] tf.float32 tensor
    trunc_mode : int (or None) : if an integer value is provided, select only
                the trunc_mode first mode given. Else if trunc_mode=None, use all modes
    freestream_target : passed through to out_nn_modes_uv (see there)
    '''
    out_NN = out_nn_modes_uv(x,y,weights,biases,geom,freestream_target=freestream_target)
    Nmode = int(out_NN[0,0,:].shape[0])
    if trunc_mode!=None and trunc_mode <= Nmode:
        Nmode = trunc_mode
    parts = [out_NN[0,:,k]*tf.exp(k*omega_0*tf.complex(0.,t[:,0])) for k in range(Nmode)]
    t_parts = tf.convert_to_tensor(parts)
    t_real = tf.real(tf.reduce_sum(t_parts,axis=0))
    return tf.transpose(tf.convert_to_tensor([t_real])) # retrait de perm=[1,0]

def NN_time_p(x,y,t,weights,biases,omega_0,trunc_mode=None):
    '''
    x,y,t : [Nint,1] tf.float32 tensors, list of coordinates (x,t) where to compute p(x,y,t)
    omega_0 : fondamental frequency
    Output [Nint,1] tf.float32 tensor
    '''
    out_NN = out_nn_modes_p(x,y,weights,biases) 
    Nmode = int(out_NN[0,0,:].shape[0])
    if trunc_mode!=None and trunc_mode <= Nmode:
        Nmode = trunc_mode
    parts = [out_NN[0,:,k]*tf.exp(k*omega_0*tf.complex(0.,t[:,0])) for k in range(Nmode)]
    t_parts = tf.convert_to_tensor(parts)
    t_real = tf.real(tf.reduce_sum(t_parts,axis=0))
    return tf.transpose(tf.convert_to_tensor([t_real]))


# =============================================================================
# Declaration of the optimisers
# =============================================================================

def declare_LBFGS(loss,maxit=50000,maxfun=50000,ftol=1.0 * np.finfo(float).eps):
    optimizer = tf.contrib.opt.ScipyOptimizerInterface(loss, method = 'L-BFGS-B', 
                                                                options = {'maxiter': maxit, #50000
                                                                           'maxfun': maxfun, #50000
                                                                           'maxcor': 50,
                                                                           'maxls': 50,
                                                                           'ftol' : ftol}) 
    print('L-BFGS-B optimizer declared with maxit = %d, maxfun = %d, ftol = %.2e'%(maxit,maxfun,ftol))
    return optimizer


def declare_Adam(loss,lr=1e-3,*args): #list_var=tf.trainable_variables()
    '''
    *args :
        var_list : trainable variables
    '''
    optimizer_Adam = tf.compat.v1.train.AdamOptimizer(learning_rate=lr)
    print('Adam optimize declared with learning rate = %.2e'%(lr))
    if len(args) == 1:
        list_var = args[0]
        return optimizer_Adam.minimize(loss,var_list=list_var)
    else:
        return optimizer_Adam.minimize(loss)


def declare_init_session():
    sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(allow_soft_placement=True, log_device_placement=False))
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)
    return sess

def square_norm(z):
    '''
    z : tf.complex64 tensor
    return tf tensor of square norm value of each complex
    '''
    return tf.square(tf.math.real(z)) + tf.square(tf.math.imag(z))

def square_norm_np(z):
    '''
    z : numpy array of complex values
    return np array of same dimension with square norm value of each complex number
    '''
    return np.square(np.real(z)) + np.square(np.imag(z))


# =============================================================================
# Entrainement
# =============================================================================


def simple_callback(loss):
    print('Loss: %.3e' % (loss))



def model_train_scipy(optimizer,sess,loss,tf_dict=None,fn_callback=simple_callback,List_loss = False, loss_valid = None, tf_dict_valid=None):
    global it
    it = 0
    global List_it_loss_LBFGS
    List_it_loss_LBFGS = []
    global List_it_loss_valid_LBFGS
    List_it_loss_valid_LBFGS = []
    
    if List_loss == True and tf_dict_valid != None:
    
        def callback(loss,List_loss=List_loss):
            global it
            global List_it_loss_LBFGS
            global List_it_loss_valid_LBFGS
            it += 1
            List_it_loss_LBFGS.append([it,loss])
            if List_loss==True and it%100 == 0:
                loss_valid_value = sess.run(loss_valid,feed_dict=tf_dict_valid)
                List_it_loss_valid_LBFGS.append([it,loss_valid_value])
            print('Loss: %.3e' % (loss))
            # global it
            # it += 1
        
        fn_callback=callback
    
    if tf_dict != None:
        optimizer.minimize(sess,
                feed_dict = tf_dict,
                fetches = [loss],
                loss_callback = fn_callback)
    else:
        optimizer.minimize(sess,
                fetches = [loss],
                loss_callback = fn_callback) 
    return List_it_loss_LBFGS,List_it_loss_valid_LBFGS

def model_train_Adam(optimizer,sess,loss,liste_tf_dict=None,Nit=1e4,tolAdam=1e-4,it=0,itdisp=1000,maxTime=None,multigrid=False,NgridTurn=1000,List_loss = False, loss_valid = None, tf_dict_valid=None):
    List_it_loss_Adam = []
    List_it_loss_valid_Adam = []
    if not(multigrid):
        tf_dict=[liste_tf_dict]
        NgridTurn=1
        Ngrid = 1
    else:
        tf_dict = liste_tf_dict
        Ngrid = len(tf_dict)
        
    t0 = time.time()
    loss_value = sess.run(loss, tf_dict[0])
    it0 = it
    conditionTime = True
    while(it-it0<Nit and loss_value>tolAdam and conditionTime):
        
        k_dict = int(it/NgridTurn)%Ngrid
        
        sess.run(optimizer, tf_dict[k_dict])
        loss_value = sess.run(loss, tf_dict[k_dict])
        
        if it%itdisp ==0:
            print('Post Adam it %d - Loss value :  %.3e' % (it, loss_value))
            
        if List_loss==True and it%100 == 0:
            loss_valid_value = sess.run(loss_valid,feed_dict=tf_dict_valid)
            List_it_loss_valid_Adam.append([it,loss_valid_value])
        List_it_loss_Adam.append([it,loss_value])
        
        
        it += 1
        conditionTime = (maxTime==None) or ((time.time()-t0)<maxTime)
        
    return List_it_loss_Adam,List_it_loss_valid_Adam




# =============================================================================
# Print and plot
# =============================================================================

def print_bar():
    print('--------------------------------------------')

def tf_print(string,tensor,sess,tf_dict=None):
    '''
    Parameters
    ----------
    string : String of character to display before the result of tf output
    tensor : tensor to compute and print
    sess : Current session object to compute given tensor
    tf_dict : dictionnary to feed, in cas it is necessary
    '''
    print(string + " " + str(sess.run(tensor,feed_dict=tf_dict)))
    


def tf_plot_scatter(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)
    
    # Step 2 : plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    plt.scatter(x_np,y_np,c=c_np,marker='.',s=1.)
    ax.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.colorbar()
    fig.tight_layout()
    
    return fig,ax


def tf_plot_scatter_complex_4fig(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    x and y are float32 tensors
    c is complex64 tensor
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)
    
    # Step 2 : plot
    fig = plt.figure(figsize=(8,6))
    plt.subplot(221)
    plt.scatter(x_np,y_np,c=np.real(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - real part')
    plt.colorbar()
    plt.subplot(222)
    plt.scatter(x_np,y_np,c=np.imag(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - imaginary part')
    plt.colorbar()
    plt.subplot(223)
    plt.scatter(x_np,y_np,c=np.sqrt(np.square(np.real(c_np))+np.square(np.imag(c_np))),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - norm')
    plt.colorbar()
    plt.subplot(224)
    plt.scatter(x_np,y_np,c=np.angle(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - angle')
    plt.colorbar()
    plt.tight_layout()
    
    return fig


def tf_plot_scatter_complex(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    x and y are float32 tensors
    c is complex64 tensor
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)
    
    # Step 2 : plot
    fig = plt.figure(figsize=(8,4))
    plt.subplot(121)
    plt.scatter(x_np,y_np,c=np.real(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - real part')
    plt.colorbar()
    plt.subplot(122)
    plt.scatter(x_np,y_np,c=np.imag(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - imaginary part')
    plt.colorbar()
    plt.tight_layout()
    
    return fig


def tf_plot_compare_3plot(x_tf,y_tf,c_tf_1,c_tf_2,sess,xlabel='x',ylabel='y',title1='',title2='',suptitle='',tf_dict=None):
    '''
    x_tf, y_tf, c_tf_1, c_tf_2 : 1D float32 tensor to compute and plot
    Return a figure with 3 subplots : c_tf_1, c_tf_2, log10((c_tf_1-c_tf_2)^2)
    '''
    
    x_np,y_np,c_np_1,c_np_2 = sess.run([x_tf,y_tf,c_tf_1,c_tf_2],feed_dict=tf_dict)
    
    fig = plt.figure(figsize=(15,4))
    plt.subplot(131)
    plt.scatter(x_np,y_np,c=c_np_1,marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title(title1)
    plt.subplot(132)
    plt.scatter(x_np,y_np,c=c_np_2,marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title(title2)
    plt.subplot(133)
    plt.scatter(x_np,y_np,c=np.log10(np.square(c_np_1-c_np_2)),marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title('Square difference - log10')
    plt.suptitle(suptitle)
    plt.tight_layout()
    
    return fig


In [ ]:
%%writefile evaluate_regions.py
"""
Standalone regional-error evaluation for a trained (pressure-only or dense)
ModalPINN run.

Loads a saved model checkpoint (no retraining) and the real CFD dataset,
reconstructs u, v, p over the real mesh nodes, and reports relative L2
error split by region (near-cylinder / near-wake / far-wake / whole domain)
so we can see whether reconstruction quality degrades away from the
sensors, per the project plan's regional-error metric (Section 8.1).

Usage:
    python evaluate_regions.py --RunDir <path to the run's output folder> \
        --WidthLayer 25 --Nmodes 3
"""
import argparse
import glob
import os
import sys

import numpy as np
import matplotlib
matplotlib.use('Agg')  # must be set before NN_functions imports pyplot
import tensorflow as tf
tf.compat.v1.disable_eager_execution()

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
import NN_functions as nnf  # noqa: E402
from text_flow import read_flow  # noqa: E402

# Geometry / physics constants, matching ModalPINN_VortexShedding.py
X_C, Y_C, R_C = 0., 0., 0.5
LXMIN, LXMAX, LYMIN, LYMAX = -4., 8., -4., 4.
GEOM = [LXMIN, LXMAX, LYMIN, LYMAX, X_C, Y_C, R_C]
OMEGA_0 = 1.036
D = 2 * R_C  # cylinder diameter

# Relative to the current working directory, matching the plain relative path
# ModalPINN_VortexShedding.py itself uses (filename_data = 'Data/fixed_cylinder_atRe100').
# Don't compute this from __file__ / repo structure - in the Colab notebook this
# script and Data/ both sit flat in /content, not nested under src/pressure_only/.
DEFAULT_DATA_FILE = 'Data/fixed_cylinder_atRe100'


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--RunDir', required=True, help="Path to the run's output folder (contains DNN..._tanh.pickle)")
    parser.add_argument('--WidthLayer', type=int, required=True)
    parser.add_argument('--Nmodes', type=int, required=True)
    parser.add_argument('--DataFile', default=DEFAULT_DATA_FILE)
    args = parser.parse_args()

    layers = [2, args.WidthLayer * args.Nmodes, args.WidthLayer * args.Nmodes, args.Nmodes]

    pickle_candidates = glob.glob(os.path.join(args.RunDir, 'DNN*_tanh.pickle'))
    assert pickle_candidates, f'No model pickle found in {args.RunDir}'
    model_file = pickle_candidates[0]
    print('Loading model:', model_file)

    w_u, b_u, w_v, b_v, w_p, b_p = nnf.restore_NN(layers, model_file, tf_as_constant=True)

    print('Reading real dataset...')
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)

    # Crop to the training domain box, same condition as read_cut_simulation_data
    nodes_x0, nodes_y0 = nodes_X[0, :], nodes_Y[0, :]
    in_box = ((nodes_x0 < LXMAX) & (nodes_x0 > LXMIN) &
              (nodes_y0 > LYMIN) & (nodes_y0 < LYMAX))
    idx = np.argwhere(in_box)[:, 0]
    nodes_x0, nodes_y0 = nodes_x0[idx], nodes_y0[idx]
    Us_c, Vs_c, Ps_c = Us[:, idx], Vs[:, idx], Ps[:, idx]

    r = np.sqrt((nodes_x0 - X_C) ** 2 + (nodes_y0 - Y_C) ** 2)
    region_near_cyl = r < 1.5 * R_C
    region_near_wake = (~region_near_cyl) & (nodes_x0 >= X_C) & (nodes_x0 < X_C + 3 * D)
    region_far_wake = (~region_near_cyl) & (~region_near_wake) & (nodes_x0 >= X_C + 3 * D)
    region_other = ~(region_near_cyl | region_near_wake | region_far_wake)

    regions = {
        'near-cylinder': region_near_cyl,
        'near-wake': region_near_wake,
        'far-wake': region_far_wake,
        'other (upstream/off-axis)': region_other,
        'whole domain': np.ones_like(region_near_cyl, dtype=bool),
    }

    x_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])
    y_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])
    t_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])

    u_pred_tf = nnf.NN_time_uv(x_tf, y_tf, t_tf, w_u, b_u, GEOM, OMEGA_0)
    v_pred_tf = nnf.NN_time_uv(x_tf, y_tf, t_tf, w_v, b_v, GEOM, OMEGA_0)
    p_pred_tf = nnf.NN_time_p(x_tf, y_tf, t_tf, w_p, b_p, OMEGA_0)

    sess = tf.compat.v1.Session()
    sess.run(tf.compat.v1.global_variables_initializer())

    Nt, Nnode = Us_c.shape
    print(f'Reconstructing {Nt} timesteps x {Nnode} nodes (one timestep at a time, to keep memory bounded)...')
    x_col = nodes_x0.reshape(-1, 1).astype(np.float32)
    y_col = nodes_y0.reshape(-1, 1).astype(np.float32)

    u_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    v_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    p_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    for k in range(Nt):
        t_col = np.full((Nnode, 1), times[k], dtype=np.float32)
        feed = {x_tf: x_col, y_tf: y_col, t_tf: t_col}
        u_pred[k, :] = sess.run(u_pred_tf, feed_dict=feed)[:, 0]
        v_pred[k, :] = sess.run(v_pred_tf, feed_dict=feed)[:, 0]
        p_pred[k, :] = sess.run(p_pred_tf, feed_dict=feed)[:, 0]
        if (k + 1) % 50 == 0 or k == Nt - 1:
            print(f'  {k + 1}/{Nt} timesteps done')

    def rel_l2(pred, true, mask):
        if mask.sum() == 0:
            return float('nan')
        diff = pred[:, mask] - true[:, mask]
        return np.linalg.norm(diff) / np.linalg.norm(true[:, mask])

    print()
    header = f"{'Region':<26}{'n_nodes':>9}{'E_u':>10}{'E_v':>10}{'E_p':>10}"
    print(header)
    print('-' * len(header))
    for name, mask in regions.items():
        eu = rel_l2(u_pred, Us_c, mask)
        ev = rel_l2(v_pred, Vs_c, mask)
        ep = rel_l2(p_pred, Ps_c, mask)
        print(f"{name:<26}{mask.sum():>9}{eu:>10.4f}{ev:>10.4f}{ep:>10.4f}")


if __name__ == '__main__':
    main()


In [ ]:
%%writefile text_flow.py
"""

Author: Mouad Boudina
From: https://zenodo.org/record/5039610


The flow file structure is the following:

Re Ur
(blank line)
Nt N_nodes (Nt = length of the timeline of the flow simulation)
(blank line)
t0
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...
t1
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...

"""
import time
import numpy as np
#==============================================================================

def floatIt(l):
    return np.array([float(e) for e in l])

def intIt(l):
    return np.array([int(e) for e in l])

def read_flow(infile):
    f = open(infile, 'r')

    t1 = time.process_time()

    print('Reading flow...')

    Re, Ur = floatIt(f.readline().strip().split())

    f.readline() # blank line

    Nt, N_nodes = intIt(f.readline().strip().split())

    f.readline()

    times = []

    nodes_X, nodes_Y = [], []
    Us, Vs, ps = [], [], []

    for n in range(Nt):
        tn = float(f.readline().strip())
        times.append(tn)

        print('%.3f' % tn)

        tmp_nodes_X, tmp_nodes_Y = [], []
        tmp_Us, tmp_Vs, tmp_ps = [], [], []

        for k in range(N_nodes):
            x, y, U, V, p = floatIt(f.readline().strip().split())

            tmp_nodes_X.append(x)
            tmp_nodes_Y.append(y)

            tmp_Us.append(U)
            tmp_Vs.append(V)
            tmp_ps.append(p)

        nodes_X.append(tmp_nodes_X)
        nodes_Y.append(tmp_nodes_Y)

        Us.append(tmp_Us)
        Vs.append(tmp_Vs)
        ps.append(tmp_ps)

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

    return Re, Ur, np.array(times), \
           np.array(nodes_X), np.array(nodes_Y), \
           np.array(Us), np.array(Vs), np.array(ps)

def write_flow(flow, outfile):
    f = open(outfile, 'w')

    t1 = time.process_time()

    print('Writing flow...')

    f.write('%.0f %.1f\n' % (flow.Re, flow.Ur))
    f.write('\n') # blank line

    Nt, N_nodes = len(flow.times), len(flow.nodes_X[0])

    f.write('%d %d\n' % (Nt, N_nodes))
    f.write('\n')

    for n in range(Nt):
        tn = flow.times[n]

        print('%.6f' % tn)

        f.write('%.6f\n' % tn)

        for k in range(N_nodes):
            f.write('%13.9f %13.9f %13.9f %13.9f %13.9f\n' %\
                    (flow.nodes_X[n, k],
                     flow.nodes_Y[n, k],
                     flow.Us[n, k],
                     flow.Vs[n, k],
                     flow.ps[n, k]))

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()


In [ ]:
%%writefile reactions_process.py
"""
Author: Mouad Boudina
From: https://zenodo.org/record/5039610
"""
import numpy as np

from scipy.optimize  import curve_fit
from scipy.integrate import trapz
#==============================================================================
labels_solid = {1:r'$X/D$',
                2:r'$Y/D$',
                3:r'$\theta$',
                4:r'$U/U_{0}$',
                5:r'$V/U_{0}$',
                6:r'$\dot{\theta}$',
                7:r'$F_{X}$',
                8:r'$F_{Y}$',
                9:r'$M_{z}$'}

labels_entity = {1:r'$F_{X}$',
                 2:r'$F_{Y}$',
                 3:r'$M_{z}$'}

labels = {'solid':labels_solid, 'entity':labels_entity}

#fontsize = 12
fontsize = 10

tol = 1e-2

def floatIt(l):
    return np.array([float(e) for e in l])

def extract_reactions(infile):
    f = open(infile, 'r')

    # A dummy test to know whether we are reading reactions of an entity or a
    # solid (in 2D).
    line = f.readline()
    if len(line.strip().split()) == 5:
        flag = 'entity'
    else:
        flag = 'solid'
    f.seek(0)

    times = []

    if flag == 'entity':
        Fx, Fy, Mz = [], [], []

        for line in f:
            # The ':-1' is used to avoid floating the name of the entity in the
            # last column.
            fragmented = floatIt(line.strip().split()[:-1])

            times.append(fragmented[0])

            Fx.append(fragmented[1])
            Fy.append(fragmented[2])
            Mz.append(fragmented[3])

        return np.array(times), np.array(Fx), np.array(Fy), np.array(Mz), flag

    else:
        X, Y, theta = [], [], []
        U, V, theta_dot = [], [], []
        Fx, Fy, Mz = [], [], []

        for line in f:
            # The ':-1' is used to avoid floating the name of the solid in the
            # last column.
            fragmented = floatIt(line.strip().split()[:-1])

            times.append(fragmented[0])

            X.append(fragmented[1])
            Y.append(fragmented[2])
            theta.append(fragmented[3])

            U.append(fragmented[4])
            V.append(fragmented[5])
            theta_dot.append(fragmented[6])

            Fx.append(fragmented[7])
            Fy.append(fragmented[8])
            Mz.append(fragmented[9])

        return np.array(times), \
               np.array(X), np.array(Y), np.array(theta), \
               np.array(U), np.array(V), np.array(theta_dot), \
               np.array(Fx), np.array(Fy), np.array(Mz), flag

def plot_reactions(reactions, variable_index, ax, color):
    plot_params = {'linestyle':'-',
                   'color'    :color,
                   'marker'   :'.',
                   'markerfacecolor':'cyan',
                   'label'    :labels[reactions[-1]][variable_index]}

    ax.plot(reactions[0], reactions[variable_index], **plot_params)

    ax.set_xlabel(r'$\bar{t}=tU_{0}/D$', fontsize=12)
    ax.set_ylabel(r'$F_{y}$', fontsize=12)

#    ax.legend(loc='best', numpoints=1,
#              fontsize=fontsize,
#              frameon=False,
#              ncol=1,
#              labelspacing=0.2,
#              handlelength=0.2)

def get_Fd_and_Fl(reactions, alpha):
    if reactions[-1] == 'solid':
        exception_msg = 'We are sorry, but this function is useful only for' \
                      + 'fixed objects.'
        raise Exception(exception_msg)

    # WARNING:
    # alpha IS IN DEGREES. MUST BE CONVERTED TO RADIANS TO USE IT IN PYTHON FUNCTIONS.
    a = (np.pi/180.)*alpha

    times, Fx, Fy = reactions[:3]

    Fd =  Fx*np.cos(a) + Fy*np.sin(a)
    Fl = -Fx*np.sin(a) + Fy*np.cos(a)

    return times, Fd, Fl

def first_maximum(reactions, variable_index, i0):
    """
    Find the first occurrence of the maximum, starting from index i0, and
    returns that index with the value of the maximum.
    """
    times = reactions[0]

    psi = reactions[variable_index]

    Nt = len(times)
    for i in range(i0, Nt-1):
        if psi[i] > max(psi[i-1], psi[i+1]):
            return psi[i], i

    raise Exception('The function is probably constant, or hasn\'t the same peak.')

def find_t400_t410(reactions):
    times = reactions[0]
    Nt = len(times)

    i400, i410 = 0, 0

    for i in range(Nt):
        if times[i] > 400.:
            i400 = i
            break

    for i in range(i400+1, Nt):
        if times[i] > 410.:
            i410 = i
            break

    return i400, i410

def find_period(reactions, variable_index, ax):
#    if reactions[-1] == 'entity':
#        exception_msg = 'We are sorry, but this function is useful only for' \
#                      + ' moving objects.'
#        raise Exception(exception_msg)

    times = reactions[0]

    i400, i410 = find_t400_t410(reactions)

    psi_max, imax = first_maximum(reactions, variable_index, i400)

    psi_max2, imax2 = first_maximum(reactions, variable_index, imax+1)
    # Sometimes there is local maximums, so we keep searching until we find
    # the real maximum that has the same value as psi_max.
    while abs(psi_max - psi_max2) > tol:
        psi_max2, imax2 = first_maximum(reactions, variable_index, imax2+1)

    psi = reactions[variable_index]

    if ax != None:
        ax.plot(times[i400:i410], psi[i400:i410], color='blue', linestyle='-')

        ax.plot(times[[imax, imax2]], psi[[imax, imax2]],
                color='red', linestyle='-', marker='o')

        ax.set_xlabel('Time', size='xx-large')
        ax.set_ylabel(labels[reactions[-1]][variable_index], size='xx-large')

    period = times[imax2] - times[imax]
    print('FLOW PERIOD = %0.6f' % period)

    mean = np.mean(psi[imax:imax2])
    print('MEAN = %0.6f' % mean)

    maxi = max(psi[imax:imax2])
    print('MAX  = %0.6f' % maxi)

    amp = maxi - mean
    print('AMP  = %0.6f' % amp)

    return period, mean, maxi, amp

def fit_a_sine(reactions, variable_index, ax):
    period, mean, maxi, amp = find_period(reactions, variable_index, ax)

    times = reactions[0]
    psi = reactions[variable_index]
    cent_norm = (psi - mean)/amp

    i400, i410 = find_t400_t410(reactions)

    def f(t, phi):
        return np.sin(2*np.pi*t/period + phi)

    popt, pcov = curve_fit(f, times[i400:i410], cent_norm[i400:i410])
    phi = popt

    print('PHASE LAG = %0.6f = %0.3f PI' % (phi, phi/np.pi))

    if ax != None:
        ax.plot(times[i400:i410], mean + amp*f(times[i400:i410], phi),
                color='r',
                linestyle='--')

    return phi

def eight_figure(reactions, frac, ax, c, equal, maxi_norm=False):
    if reactions[-1] == 'entity':
        exception_msg = 'The eight is a trajectory of a moving solid, but your' \
                      + ' entry is an entity (i.e. fixed body).'
        raise Exception(exception_msg)

    times, X, Y = reactions[:3]

    Nt = len(times)

    # Same notice as in the previous function find_period.
    i0 = int(frac*Nt)

    x, y = X[i0:]/0.075, Y[i0:]
#    x, y = X[i0:], Y[i0:]

    width  = max(x) - min(x)
    height = max(y) - min(y)
    ratio  = height/width

    print('width/2 = %0.6f' % (width/2.))
    print('ratio   = %0.6f' % ratio)

    m = np.mean(x)
    
    if maxi_norm:
        maxi_x = np.max(x-m)
        maxi_y = np.max(y)
    else:
        maxi_x = 1
        maxi_y = 1

    ax.plot((x - m)/maxi_x + c, y/maxi_y, linestyle='-', color='black')

#    ax.set_xlabel(r'$X/D$', fontsize=fontsize)
    ax.set_xlabel(r'$U_{\mathrm{r}}$', fontsize=fontsize)
    ax.set_ylabel(r'$Y/D$', fontsize=fontsize)
    
    ax.tick_params(axis='both', which='major', labelsize=fontsize)

#    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

    ax.set_xticks(range(3,11))
#    ax.set_yticks([-5e-1,0,5e-1])

    if equal:
        ax.axis('equal')

    annotate = False
    if annotate:
        x1 = min(x) - m
        x2 = max(x) - m

        y1_arrow = 1.1*min(y)
        y1_text = 1.1*y1_arrow

        ax.annotate(s='', xy=(x1,y1_arrow), xytext=(x2,y1_arrow),
                    arrowprops=dict(arrowstyle='<->'))

        ax.text(x= (x1 + x2)/2.,
                y=y1_text,
                s=r'$2\bar{X}_{\mathrm{max}}$',
                color='black',
                horizontalalignment='center',
                verticalalignment='top',
                fontsize=fontsize)

#    ax.set_ylim([-.9,.9])
#    ax.set_ylim([-.7,.7])
#    ax.set_xlim([-.09,.09])

def find_upper_shell(reactions, frac, ax, equal):
    if reactions[-1] == 'entity':
        exception_msg = 'The upper shell is the upper trajectory of a moving '\
                      + 'solid, but your entry is an entity.'
        raise Exception(exception_msg)

    times, X, Y = reactions[:3]
    U, V = reactions[4:6]

    speed = np.sqrt(U**2 + V**2)

    Nt = len(times)

    i0 = int(frac*Nt)
#    x, y = X[i0:], Y[i0:]

    imin, imax = 0, 0

    # Countercurrent at outer shell
#    for i in range(i0, Nt-1):
#        if X[i] > max(X[i-1], X[i+1]):
#            imax = i
#            break
#
#    for i in range(imax + 1, Nt-1):
#        if X[i] < min(X[i-1], X[i+1]):
#            imin = i
#            break
#
#    ax.plot(X[imax:imin+1], Y[imax:imin+1], 'oc', linewidth=2)

    # Countercurrent at inner shell
    for i in range(i0, Nt-1):
        if X[i] < min(X[i-1], X[i+1]):
            imin = i
            break

    for i in range(imin + 1, Nt-1):
        if X[i] > max(X[i-1], X[i+1]):
            imax = i
            break

    ax.plot(X[imin:imax+1], Y[imin:imax+1], color='cyan', marker='o', linewidth=2)

#    imax_speed = imin
#    for i in range(imax + 1, Nt-1):
#    for i in range(imax + 1, imin-1):
#        if speed[i] > max(speed[i-1], speed[i+1]):
#            imax_speed = i
#            break

#    ax.plot(X[imax:], Y[imax:], color='black', linewidth=0.5)
    ax.plot(X[i0:], Y[i0:], color='black', linewidth=0.5)

#    ax.plot([X[imax_speed]], [Y[imax_speed]], color='red', marker='o',
#            label='Position of maximum speed')

    ax.plot([X[i0]], [Y[i0]], color='black', marker='o',
            label='Starting point')
    ax.plot([X[i0+5]], [Y[i0+5]], color='gray', marker='o')

#    ax.legend(loc='best', fontsize=12, numpoints=1)

    if equal:
        ax.axis('equal')

    tg = abs(Y[imax]/Y[imin])
    p = (2./np.pi)*np.arctan(tg)
    print('p = %.6f' % p)

#    max_velocity = max(speed[imax:imin+1])
#    print('||Umax|| = ' + str(max_velocity))

def travel_length(Xmax, Ymax, p):
    AR = Ymax/Xmax

    zeta = np.linspace(-.999, .999, 100)

#    tmp = np.sin(p*np.pi/2.)/np.sqrt(1 + zeta) - np.cos(p*np.pi/2.)/np.sqrt(1 - zeta)
    tmp = np.sin(p*np.pi/2.)/np.sqrt(1 + zeta) + np.cos(p*np.pi/2.)/np.sqrt(1 - zeta)

    tmp *= (1/8.)*AR

    integrand = np.sqrt(1 + tmp**2)

    integral = trapz(integrand, zeta, axis=0)

    return Xmax*integral



## 7. Get the real dataset (Boudina et al., Zenodo, ~1.17 GB)

Cached in Drive after the first download (from the E3 run), so this should just copy locally instead of re-downloading from Zenodo.

In [ ]:
import os, shutil
os.makedirs('Data', exist_ok=True)
drive_cache = '/content/drive/MyDrive/ModalPINN_data/fixed_cylinder_atRe100'
local_path = 'Data/fixed_cylinder_atRe100'
if os.path.exists(drive_cache):
    print('Found cached dataset in Drive, copying locally...')
    shutil.copyfile(drive_cache, local_path)
else:
    print('Not cached yet, downloading from Zenodo...')
    exit_code = os.system(
        'curl -L -o ' + local_path +
        ' "https://zenodo.org/records/5039610/files/fixed_cylinder_atRe100?download=1"')
    assert exit_code == 0, 'Download failed'
    os.makedirs(os.path.dirname(drive_cache), exist_ok=True)
    shutil.copyfile(local_path, drive_cache)
    print('Saved a copy to Drive for future runs:', drive_cache)
print('Dataset ready:', local_path, os.path.getsize(local_path), 'bytes')


## 8. Run the experiment

Identical to E3 (`--SparseData --PressureOnly --NTaps 32 --Seed 0 --Tmax 9 --Nmes 5000 --Nint 50000 --multigrid --Ngrid 5 --NgridTurn 200 --WidthLayer 25 --Nmodes 3`), plus `--FreestreamBC` - the only difference from E3, so any change in the regional evaluation numbers can be attributed to this one flag.

This cell runs in the foreground and can take up to ~9-10 hours. Once it's printing `Loss:` lines, you can close this tab - with Colab Pro+, background execution keeps this notebook's runtime running, and with **Run all**, the Drive-copy and evaluation cells fire automatically the moment this cell finishes, so results are safe even if you're not watching when it completes.

In [ ]:
import os
os.environ['LD_LIBRARY_PATH'] = '/content/miniconda/envs/modalpinn/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
!/content/miniconda/envs/modalpinn/bin/python ModalPINN_VortexShedding.py \
    --SparseData --PressureOnly --NTaps 32 --FreestreamBC --Seed 0 \
    --Tmax 9 --Nmes 5000 --Nint 50000 \
    --multigrid --Ngrid 5 --NgridTurn 200 \
    --WidthLayer 25 --Nmodes 3


## 9. Copy results to Drive and verify
Runs automatically right after training finishes (if you used Run all).

In [ ]:
import glob, shutil, os, datetime
runs = sorted(glob.glob('OutputPythonScript/ModalPINN_*'))
print('Found run folders:', runs)
assert runs, 'No output folder found - training may not have finished'
latest = runs[-1]
run_name = 'E3_freestream_pressure_only_Re100_Nm3_Nint50000_Nmes5000_WL25_Ntap32_FSBC_seed0_' + datetime.date.today().strftime('%Y%m%d')
dest = os.path.join('/content/drive/MyDrive/ModalPINN_results', run_name)
shutil.copytree(latest, dest, dirs_exist_ok=True)
print('Original folder:', latest)
print('Copied to      :', dest)
print('Verifying by listing Drive path contents:')
for f in os.listdir(dest):
    full = os.path.join(dest, f)
    print(' -', f, os.path.getsize(full), 'bytes')


## 10. Confirm by reading a file back from Drive

In [ ]:
with open(os.path.join(dest, 'out.txt')) as f:
    content = f.read()
print('Read', len(content), 'bytes from Drive-backed file')
print(content[-2000:])


## 11. Regional accuracy evaluation

Same regional breakdown as E3 - compare the near-cylinder E_u/E_v numbers directly against E3's (1.49 / 1.99) to see whether the free-stream prior actually helped.

Runs against `latest` (local copy) rather than `dest` (Drive) to avoid the Drive FUSE-mount sync race - see E3's notes for why.

In [ ]:
import subprocess
r = subprocess.run(
    ['/content/miniconda/envs/modalpinn/bin/python', 'evaluate_regions.py',
     '--RunDir', latest, '--WidthLayer', '25', '--Nmodes', '3'],
    capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-3000:])
output_text = r.stdout + ('\n--- stderr ---\n' + r.stderr if r.returncode != 0 else '')
with open(os.path.join(dest, 'regional_evaluation.txt'), 'w') as f:
    f.write(output_text)
print('Saved evaluation output to', os.path.join(dest, 'regional_evaluation.txt'))
